# Module 2 · Applications: a classifier on CESM2 LENS

**Does the autumn ENSO state predict a wet Front Range winter?** A supervised classification built on the NSF NCAR-distributed
CESM2 Large Ensemble (made by NSF NCAR with the IBS Center for Climate Physics, South Korea), following the analysis spine of NSF Unidata's *Supervised Machine Learning Readiness*. 

[![Launch this notebook on Binder (binder.projectpythia.org)](https://binder.projectpythia.org/badge_logo.svg)](https://binder.projectpythia.org/v2/gh/glenbuchanan/cesm2-lens-ml-readiness-cookbook/main?labpath=notebooks%2F02_applications.ipynb) — opens the cookbook repository's copy of this notebook (`notebooks/02_applications.ipynb`, branch `main`), built from that repository's `environment.yml`, whose pins for every library this notebook imports equal `course/environment.yml` here (the course file also pins keras and pytorch for Module 3). 

*Code: Apache-2.0 · Content: CC BY 4.0 · Data: NCAR CESM2 Large Ensemble Cloud-Optimized Dataset (Grover & Banihirwe 2021, https://doi.org/10.26024/y48t-q717) and Rodgers et al. 2021 (https://doi.org/10.5194/esd-12-1393-2021), UCAR Data Terms of Use; Community Earth System Model v2 Large Ensemble (CESM2 LENS) was accessed on 2026-10-20 from https://registry.opendata.aws/ncar-cesm2-lens.*
*Drafted with AI assistance (Anthropic's Claude) from cited sources, under the author's direction; the author's line-by-line edit and a subject-matter review are still to come.*


## Overview

You build a classifier that reads one number — the autumn Niño 3.4 state of a CESM2 LENS member — and says whether that member's coming Front
Range winter will be wet, on the 100 members of the NSF NCAR-distributed large ensemble. At the end you decide, in writing, whether the model
that survived the refinement trials — or the simplest rule it had to beat — deserves to be used, justified as Unidata's closing instruction
asks: "using data and a robust evaluation". 

Objectives: 

1. **O2** — the steps of a supervised analysis and the failure each step guards against: framing → data → split → train → validate → refine → test → justify.
2. **O3** — build and validate a classification model on a climate-model ensemble, reporting precision, recall and a confusion matrix, and justify the choice honestly, *including when the simplest wins*.

**What to expect.** The signal is weak, and you will measure how weak from the printed lines, not from this text. In 3d the
Niño-sign rule's precision sits below the base rate; in 3f the logistic rung moved to the base-rate threshold is promoted over that
rule on the uncorrected test only, and is not itself clearly above the base rate. Whether a *nonlinear* rung at the same threshold
does better is the obvious next trial, and the register leaves room for it; the held-out test (3g) is yours to run, once. Four
limits frame whatever you find: this is a **model-world** study (CESM2 winters scored against CESM2 winters, never against
observations); its ENSO and the teleconnection that carries ENSO to Colorado are CESM2's own; the ~1° grid smooths the Front Range
orography through which Stagge et al. (2023) find the ENSO signal is amplified; and the shipped season, DJF, is not the one NWS
Boulder names for El Niño's wet signal east of the Divide ("particularly in the late winter and spring"). A result here is not a
result for the real Front Range. 

The spine is NSF Unidata's (Apache-2.0), with one step changed on purpose: Unidata splits *randomly*;
here the split is **by ensemble member**, and 2c says why. 

**Exercises 1–11.** Each exercise box here is a one-line pointer to its *Handbook entry* (`handbook.md`); the Handbook prompt, with its
evidence list, is the prompt of record and what the *Rubric* (`rubric.md`) scores, first by you, then by the trainer. 


## Prerequisites

| Concept | Importance | Notes |
|---|---|---|
| Python at the level of the [Xarray tutorial](https://tutorial.xarray.dev/) | Necessary | a Zarr store, a box selection, `groupby` |
| Module 1 · Foundations (no-code lesson) | Necessary | the spine and the step → failure table (O2) |
| Module 1's closed-form rule | Helpful | why "does a simpler solution exist?" is asked first |
| scikit-learn | Not required | every call is introduced where it is used |

- **Time to learn.** Reading — this notebook's prose plus the Handbook's Module 2 section, whose prompts you read to answer: about 12,788 words at 150 words a minute: about 85 minutes (the rubric is consulted, not read through; the count includes the Handbook's response scaffolds, the tables you fill). Running — every cell in about two minutes on a laptop from the committed cache (the one-time derivation is a background job: the course's one full run took 33 minutes on a fast link; on home broadband expect an hour or more). Writing — the eleven entries, not yet timed with a learner. The course records this module as about one hour *as designed*; the measured length stands beside it, here. 
- **Environment:** `environment.yml` (beside this notebook; conda-forge, pinned): `conda env create -f environment.yml && conda activate ncar-ml-readiness`; the versions this run resolved are printed under Imports; it is Module 2's environment file.
- **Where it runs:** a laptop, from the AWS copy of CESM2 LENS (anonymous S3, free egress), or the NSF NCAR JupyterHub; no local paths, no account. **Network:** 2a tries to read the catalogue and one store's metadata (about 1 MB) even when the cache exists; with no route to AWS it prints `OFFLINE` and runs on from the cache (the Metadata Document's catalogue and store rows are then *not printed this run*, Entry 3), and every later cell runs from the cache.
- **Export control:** none — open data, open code.


## Imports


In [ ]:
import re
import sys
import itertools
import warnings
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, StratifiedKFold, FixedThresholdClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score,
                             accuracy_score, average_precision_score, classification_report)

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.dpi"] = 110
# The course's figure surfaces: warm canvas, muted axes, one gold
# ramp for the emphasised element. The colour-blind encoding stays: in 2b dry/wet are teal circles AND magenta triangles (colour and shape).
plt.rcParams.update({"figure.facecolor": "#FBFAF6", "axes.facecolor": "#F5F3ED", "axes.edgecolor": "#E4E0D6", "axes.labelcolor": "#13161A",
                     "text.color": "#13161A", "xtick.color": "#6A7077", "ytick.color": "#6A7077", "grid.color": "#E4E0D6",
                     "legend.facecolor": "#F6F1E5", "legend.edgecolor": "#E4E0D6", "savefig.facecolor": "#FBFAF6"})
GOLD = ["#8F7022", "#B98E33", "#C9A23A", "#D2AD45", "#DDBC63", "#E2C168", "#E7CD82", "#EAD27E"]
from matplotlib.colors import LinearSegmentedColormap
GOLD_CMAP = LinearSegmentedColormap.from_list("gold", ["#FBFAF6", "#EAD27E", "#C9A23A", "#8F7022"])

ACCESS_LINE = ("Community Earth System Model v2 Large Ensemble (CESM2 LENS) was accessed on "
               "ACCESS_DATE from https://registry.opendata.aws/ncar-cesm2-lens.")   # the registry's access note (date + location)
# Under every LENS figure (the registry page's "How to cite"): the credit in COMET's slash-combined remix style and the
# two DOIs the UCAR Data Terms bind users to — the AWS subset (Grover & Banihirwe 2021) and the paper (Rodgers et al. 2021) — then the access line.
# The author element carries © and the year: it names who grants the figure's CC BY 4.0 (combined credit).
LENS_CITATION = ("© 2026 Glen Buchanan / CESM2 Large Ensemble Community Project (Rodgers et al. 2021, doi:10.5194/esd-12-1393-2021) / "
                 "NSF NCAR CISL, CESM2 LENS on AWS (Grover & Banihirwe 2021, doi:10.26024/y48t-q717) · figure CC BY 4.0\n" + ACCESS_LINE)

def cite(fig):
    """Put the required CESM2 LENS citation line under a figure."""
    fig.text(0.01, -0.02, LENS_CITATION, fontsize=7, ha="left", va="top")

def credit_own(fig, what):
    """Under a course diagram that draws no CESM2 LENS data (the two boxes in Part 1; the spine in the Summary): who grants its CC BY 4.0,
    and what it is — the LENS citation would claim a data source the figure does not read."""
    fig.text(0.01, -0.02, f"© 2026 Glen Buchanan · figure CC BY 4.0 · {what}", fontsize=7, ha="left", va="top")


In [ ]:
# The environment record: the versions THIS run resolved, read from the installed packages — never typed in.
# environment.yml pins them; if a line below disagrees with it, the kernel is not the course environment.
from datetime import datetime
print(f"run started {datetime.now().astimezone().isoformat(timespec='seconds')}   kernel python {sys.version.split()[0]}")
PACKAGES = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "xarray", "zarr", "s3fs", "cftime", "intake", "intake-esm"]
print(f"python        {sys.version.split()[0]}")
for pkg in PACKAGES:
    try:
        print(f"{pkg:13s} {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:13s} NOT INSTALLED — environment.yml pins it; re-create the environment")


### The decisions this notebook records

Every value below is a **design decision**, not a default, recorded in the Metadata Document (2a) with its reason so the analysis can be rerun
(O2's *train* row). The seed is declared once, at the top, and passed down — never `np.random.seed()` (scikit-learn's *Common pitfalls and recommended practices*, §12.3.3). 


In [ ]:
RNG_SEED = 0                                    # declared once; every random_state below derives from it (2b creates its RandomState from it right before the test draw)

ACCESS_DATE = "2026-10-20"                      # the day the COMMITTED data/member_winters.csv was derived from s3://ncar-cesm2-lens. 2a asserts it equals the cache
                                                #   header's access= field; a REBUILD = True run overwrites it with its own date, so the header and every figure citation
                                                #   name the day the data were actually read (never the committed date under a rebuilt table).
LENS_TEMPLATE = LENS_CITATION                   # keeps the ACCESS_DATE token: 2a re-stamps the citation from it after a rebuild
LENS_CITATION = LENS_TEMPLATE.replace("ACCESS_DATE", ACCESS_DATE)

# Which values are yours: SEASON (Exercise 1c) is YOUR decision — DJF is the shipped path (the committed cache); FMA costs a rebuild (below).
# Every other value below is a course design decision, recorded with its reason in the Metadata Document (2a); change one only as a
# refinement trial you record, never silently.

# --- the target ------------------------------------------------------------------------------------------
FRONT_RANGE = dict(lat=slice(38.5, 40.5), lon=slice(253.75, 256.25))   # course design: Denver–Fort Collins and the Front Range, 0–360 grid; three 1.25° columns, the westernmost (centre 253.75E = 106.25W) straddling the Continental Divide
SEASON = "DJF"      # YOUR DECISION (Exercise 1c): "DJF" (Dec of the prior year + Jan, Feb) or "FMA" (Feb–Apr). DJF is the shipped path:
                    #   the committed cache was derived for it. FMA needs REBUILD = True in 2a (the cache cell refuses a cache derived
                    #   with another season): reading roughly 160 GB of global chunks — the course's one full derivation took 33 minutes on a fast
                    #   link (2026-10-20); on home broadband expect an hour or more (data/README.md).
                    #   NWS Boulder's 2020 winter-outlook slides: east of the Divide the El Niño wet signal
                    #   is mainly late winter–spring, so the season you score changes how much signal exists to learn.
WET_QUANTILE = 2/3  # course design: "wet" = detrended winter precipitation above this quantile of the TRAINING members, pooled.
                    #   ONE threshold for every member, fit on training members only (2b; refit inside every validation fold, 2c)
                    #   and applied to held-out members as a transform. A per-member quantile would fix every member's base rate
                    #   at 1 - WET_QUANTILE by construction, and the per-half check in Exercise 4 could never fail.

# --- the input -------------------------------------------------------------------------------------------
NINO34 = dict(lat=slice(-5, 5), lon=slice(190, 240))   # 5°S–5°N, 170°–120°W: the Rodgers et al. 2021 box, on atm TS
AUTUMN_MONTHS = [9, 10, 11]                            # SON of the year before the winter; recorded in the cache header and asserted by 2a like SEASON (a changed autumn needs a rebuild)

# --- the window and the forced response ------------------------------------------------------------------
WINDOW = (1900, 2014)   # course design: the first winter. Rodgers et al. 2021 §2.2: members "should not be considered …
                        #   independent" for decades after 1850 (initial-condition memory), so the window starts late.
FORCED_DEG = 3          # course design default (Metadata Document): the forced response of each series is a degree-FORCED_DEG
                        #   polynomial in year fitted to the ensemble mean of the TRAINING members, subtracted from every member
                        #   before the index and the target are formed (2b) — otherwise every member follows the same warming path
                        #   by calendar year and the models learn time-of-century, read as ENSO skill. 2b checks the removal with
                        #   the per-decade mean of the anomalies against a stated tolerance (a misfit curve fails it), so the choice
                        #   is checked, not assumed.

# --- the split -------------------------------------------------------------------------------------------
TEST_FRACTION = 0.2     # of EACH half's members, held out for the one final test (3g); named in 2b before any statistic is fit
N_FOLDS_MAX = 5         # member-grouped folds for validation; the count follows the data: N_FOLDS = min(N_FOLDS_MAX, training members), set in 2c
N_REPEATS = 10          # fold re-draws per refinement trial (3f); collapses to 1 when every re-draw is the same partition (2c says when)

# --- the ladder's settings (one home: 2c's contrast cell and 3a's ladder both read them) -------------------
TREE_SETTINGS = dict(max_depth=3, min_samples_leaf=20)
BOOSTING_SETTINGS = dict(n_estimators=100, max_depth=2, learning_rate=0.05)


## Part 1: Problem framing

Unidata's notebooks open every analysis with three questions (Apache-2.0, NSF award #2319979).
The framing step guards against O2's first failure: *a target that does not answer the decision anyone will make.* 

The three questions are asked of you first. Write your own answers in the Handbook (Entry 2) before you open the course's, which sit folded under the figure below (click *The course's three answers* to unfold them) — an answer read before it is written measures copying, not framing. 

:::{note}
**Exercise 1 — the expected sign, before fitting anything.** After the `run started …` line has printed under Imports and before any data cell runs: the sign you expect, why `SEASON` changes how much signal there is to learn, and your `SEASON` choice with its reason and its cost. → **Handbook entry 1**
:::

:::{note}
**Exercise 2 — the three framing answers.** Write yours first — the course's answers are folded under the figure and stay closed until you open them: the three simple solutions (the Niño-sign rule among them as the physical prior, no bar), whether the requirements can be met (the physics half now; the data half after 2a has printed), and the question as a decision in one line of your own. Then unfold the course's three answers and note where yours differ. → **Handbook entry 2**
:::


In [ ]:
# The two boxes the question is asked over, read from the parameter cell (one home for their values). No CESM2 LENS data is opened here —
# 2a does that: the figure draws the declared boxes on the 0–360° longitude grid the stores use, so that "autumn Niño 3.4 → Front Range
# winter" is a picture before it is a table. No coastlines: cartopy is not in environment.yml, and the boxes are the point.
def span(box):
    """A box as degrees with hemispheres, from its lat/lon slices on the 0–360 grid."""
    y0, y1, x0, x1 = box["lat"].start, box["lat"].stop, box["lon"].start, box["lon"].stop
    lat = lambda y: f"{abs(y):g}°{'S' if y < 0 else 'N'}"
    lon = lambda x: f"{x:g}°E" if x <= 180 else f"{360 - x:g}°W"
    return f"{lat(y0)}–{lat(y1)}, {lon(x0)}–{lon(x1)}"

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.set(xlim=(165, 275), ylim=(-24, 56), ylabel="latitude")
ax.set_title(f"The question's two boxes: the autumn (SON) Niño 3.4 state as the input, {SEASON} Front Range precipitation as the target", fontsize=9)
xt = list(range(170, 276, 15))
ax.set_xticks(xt); ax.set_xticklabels([f"{x}°E" if x <= 180 else f"{360 - x}°W" for x in xt], fontsize=8)
ax.set_yticks(range(-20, 61, 20)); ax.set_yticklabels([f"{abs(y)}°{'S' if y < 0 else 'N'}" for y in range(-20, 61, 20)], fontsize=8)
ax.axhline(0, lw=0.8, c="#6A7077", ls=":"); ax.grid(True, lw=0.5)
for name, box, note, at, va in [("NINO34", NINO34, "input: SON mean of TS (Rodgers et al. 2021 box)", (215, -13), "top"),
                                ("FRONT_RANGE", FRONT_RANGE, f"target: {SEASON} mean of PRECC + PRECL (Front Range box: the Divide to the plains)", (236, 50), "bottom")]:
    x0, x1, y0, y1 = box["lon"].start, box["lon"].stop, box["lat"].start, box["lat"].stop
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor="#C9A23A", edgecolor="#8F7022", lw=1.2, zorder=3))
    ax.annotate(f"{name}  {span(box)}\n{note}", xy=((x0 + x1) / 2, y0 if va == "top" else y1), xytext=at, fontsize=7.5, ha="center", va=va,
                arrowprops=dict(arrowstyle="-", color="#6A7077", lw=0.8))
ax.annotate("", xy=(FRONT_RANGE["lon"].start, FRONT_RANGE["lat"].start), xytext=(NINO34["lon"].stop, NINO34["lat"].stop),
            arrowprops=dict(arrowstyle="->", color="#8F7022", lw=1.6, connectionstyle="arc3,rad=-0.25"))
ax.text(243, 22, f"SON of year y − 1\n→ {SEASON} winter y", fontsize=8, ha="right", va="center", color="#8F7022")
credit_own(fig, "a diagram of the parameter cell's boxes; no CESM2 LENS data drawn"); plt.tight_layout()


<details>
<summary><b>The course's three answers — open after Entry 2's first draft is written</b> (folded in the rendered notebook (JupyterLab) so that yours are written first; a plain-text reader of the .ipynb sees them open — write yours first anyway; the data half of the second answer is checked against 2a's printed lines once they exist) </summary>

**Does a simpler solution exist?** Three do — the table below: two bars and the physical prior, which 3a scores and Entry 6 records. None is
Module 1's closed-form case: no known relation gives winter precipitation from autumn sea-surface temperature exactly. The relation is statistical and, by the published evidence, weak — so a model is measured
against the three rules rather than assumed to beat them, and a rule that sits below the base-rate bar, however physical, is not the simplest
solution; the base rate itself is. 

| Simple solution | What it predicts | The bar it sets, or the prior it is (3a's rung 0; Entry 6 scores it) |
|---|---|---|
| always-dry | "dry" for every winter | accuracy = 1 − base rate, about two thirds; wet precision 0/0 (undefined; scored 0 with `zero_division=0`) and recall 0 (no wet winter is caught) — a bar on accuracy only |
| always-wet | "wet" for every winter | recall 1 at precision = the base rate, about one third — the bar every model's precision is read against |
| Niño-sign rule | "wet" when the autumn Niño 3.4 anomaly is positive | the physical prior (NWS Boulder's sign east of the Divide: El Niño wetter, La Niña drier): the rule a model must beat — recorded as *physical prior — no bar* on any metric, the words Entry 6's row uses |

**Can machine learning requirements be met?** On the data side, yes: the catalogue holds monthly PRECC, PRECL and TS for every historical member
in both forcing variants — 100 members × 115 winters with a measurable target and a measurable input. On the physics side the honest prior is a
*small* effect: Stagge et al. find the ENSO winter-precipitation signal over the Intermountain West is a north–south dipole, "the northern pole …
north of 40° N" and "the southern pole … south of 35° N", with the Front Range in the transition zone between them, and Rodgers et al. 2021
report no Front Range value for CESM2 LENS. The signal is something this notebook **measures**, not assumes — and if it is weak, the simplest
rule winning is the honest answer, not a failure. 

**Which scientific question should be answered?** *"Given the autumn Niño 3.4 state in a CESM2 LENS member, will that member's coming Front Range
winter be wet — above the one pooled threshold that marks the upper third of the training members' detrended winters?"* Written as a decision:
the person who acts on it is the forecaster drafting the autumn seasonal outlook for the Front Range — the deck NWS Boulder and NWS Pueblo issue
each September — who must lean the winter outlook wet, dry, or leave it at "equal chances". A "wet" call right more often than the base rate
without missing most wet winters would move that lean; a model that cannot beat the sign rule leaves it where the office already puts it.
 

</details>

:::{note}
**The committed cache is the full table** — `data/member_winters.csv` was derived with `N_MEMBERS_PER_HALF = 50`, and 2a asserts the parameters it was derived with. A cache rebuilt with fewer members runs the same code; the fold count, the repeats and the statistical power then follow that sample.
:::


## Part 2: Data handling

### 2a Locate the data — the Metadata Document

The *data* step guards against O2's second failure: *inputs whose origin and quality were never recorded.* Unidata's Metadata Document has
four parts — **general information · structure · quality · provenance** — filled here from the catalogue itself, not from memory. 

The CESM2 LENS intake-esm catalogue holds 322 Zarr stores. Two facts it forces on the design: there is **no
monthly PRECT**, so the target is `PRECC + PRECL`; and there is **no variable named SST** (and no ocean `TEMP` for the historical-smbb half), so
the Niño 3.4 index is built from atmospheric `TS` over open ocean. 


In [ ]:
import intake

CATALOG_URL = "https://ncar-cesm2-lens.s3-us-west-2.amazonaws.com/catalogs/aws-cesm2-le.json"
# Offline (a train; a hub with no route to AWS): everything after this cell runs from the committed cache, so an unreachable catalogue is
# reported, not raised — the Metadata Document rows this cell would print are then *not printed this run* (Entry 3 says how to mark them).
try:
    cat = intake.open_esm_datastore(CATALOG_URL)
    subset = cat.search(component="atm", frequency="monthly", experiment="historical", variable=["PRECC", "PRECL", "TS"])
    print(f"catalogue last_updated: {cat.esmcat.last_updated}   rows in catalogue: {len(cat.df)}   rows selected: {len(subset.df)}")
    # The six store paths in full: the Metadata Document's "Stores used" row says they are printed here, so pandas may not truncate them.
    with pd.option_context("display.max_colwidth", None):
        display(subset.df[["variable", "forcing_variant", "units", "start_time", "end_time", "path"]])
except Exception as err:                                   # no network, a DNS failure, S3 unreachable
    cat = subset = None
    print(f"OFFLINE: the catalogue at {CATALOG_URL} could not be read ({type(err).__name__}). The Metadata Document's catalogue and store rows "
          "are *not printed this run*; every later cell runs from data/member_winters.csv.")

In [ ]:
# Open one store lazily to record its structure in the Metadata Document (nothing is downloaded yet).
try:
    probe = xr.open_zarr("s3://ncar-cesm2-lens/atm/monthly/cesm2LE-historical-cmip6-TS.zarr",
                         consolidated=True, storage_options={"anon": True})
except Exception as err:                                   # offline: the structure rows are not printed this run
    probe = None
    print(f"OFFLINE: the TS store could not be opened ({type(err).__name__}); the dims, calendar and chunking rows are *not printed this run*, "
          "and the cache cell below reads the member count from the cache header instead of the store.")
if probe is not None:
    print("dims      :", dict(probe.sizes))
    print("coords    :", list(probe.coords))
    print("members   :", probe.member_id.values[:3], "…", "calendar:", type(probe.time.values[0]).__name__)
    # The structural number the Metadata Document records, computed from the store just opened — never typed in. A Zarr chunk is read whole,
    # and every chunk is stored at full chunk shape (the last one along time padded), so a box subset of one member-variable reads every chunk.
    ch = tuple(int(c) for c in probe["TS"].encoding["chunks"])
    per_chunk = int(np.prod(ch)) * probe["TS"].dtype.itemsize                 # bytes, uncompressed (Blosc on the wire)
    n_time_chunks = -(-probe.sizes["time"] // ch[1])                          # chunks along time per member
    per_member_var = n_time_chunks * per_chunk
    n_members = 2 * probe.member_id.size                                      # two halves, each the size of this store
    print(f"chunking  : {ch} {probe['TS'].dtype} → {per_chunk / 1e6:.0f} MB per chunk uncompressed; one member-variable ({probe.sizes['time']} months) "
          f"= {n_time_chunks} chunks × {per_chunk / 1e6:.0f} MB ≈ {per_member_var / 1e6:.0f} MB read for any box subset; "
          f"the derivation ({n_members} members × 3 variables) ≈ {n_members * 3 * per_member_var / 1e9:.0f} GB")

#### The Metadata Document

Every cell below is filled from the catalogue output and the store attributes printed above, with the parameter cell for the derived rows;
the evidence column names where each fact comes from; the Window row rests on Rodgers et al. 2021 §2.2.    

| Part | Field | Value | Evidence |
|---|---|---|---|
| General | Dataset | CESM2 Large Ensemble on AWS Open Data, historical experiment, 1850–2014 | AWS Open Data registry page |
| General | Licence and citation | UCAR Data Terms of Use (CC BY 4.0), which bind users to NSF NCAR's recommended citation and access line — written out in Additional resources | AWS Open Data registry page, How to cite |
| General | Acknowledgement | the CESM2 Large Ensemble Community Project and supercomputing resources provided by the IBS Center for Climate Physics in South Korea | CESM2 LE project page |
| Structure | Stores used | six: PRECC, PRECL, TS × forcing variants cmip6, smbb (the six paths printed in full above) | catalogue JSON (`aws-cesm2-le.json`, the file 2a opens) |
| Structure | Grid, calendar, units | 192 × 288 (≈1°), monthly, noleap calendar; PRECC/PRECL m/s → mm/day × 8.64e7; TS in K; one chunk = one member × 600 months × the full grid, read whole (the `chunking` line computes the MB) | store attrs (cell above) |
| Structure | Members | 100 = 50 cmip6 + 50 smbb; initialisation groups 10 + 80 + 10; 20 micro-members per AMOC start date | CESM2 LE project page; Rodgers et al. 2021 §2.2 |
| Structure | Derived variables | PRECT = PRECC + PRECL over `FRONT_RANGE`, `SEASON` mean; Niño 3.4 = SON mean of TS over `NINO34`; both minus the **forced response** (2b) | this notebook, 2a-derive and 2b |
| Structure | One row = | one member-winter; target `wet` = detrended PRECT above **one** threshold, the `WET_QUANTILE` of the pooled training members (2b) | this notebook |
| Quality | Known gap | ocean TEMP absent for historical-smbb → TS used for both halves | catalogue JSON (`aws-cesm2-le.json`, the file 2a opens) |
| Quality | The two halves | smbb = smoothed biomass-burning forcing 1990–2020 + two minor bug fixes; **check the base rate per half before pooling (Exercise 4)** | CESM2 LE project page |
| Quality | Window | first winter **1900** (`WINDOW[0]`): Rodgers et al. 2021 §2.2 say the members "should not be considered … independent" for the years immediately after 1850 and that the micro-clusters' AMOC memory converges "only after several decades", so the window opens fifty years in; 2c measures what is left (the sibling line). Forced trend: **removed** as a polynomial in year of degree `FORCED_DEG` (`FORCED_DEG = 3` in the parameter cell; the first row of the design-defaults table below) fitted to the *training* members' ensemble mean; 2b checks the removal | Rodgers et al. 2021 §2.2; parameter cell |
| Provenance | Catalogue | `aws-cesm2-le.json`, last_updated 2021-12-27T14:29:25Z; documentation DOI 10.26024/y48t-q717 | catalogue JSON; registry |
| Provenance | Access | anonymous S3, us-west-2, `storage_options={"anon": True}`; access date 2026-10-20 for the committed cache (`ACCESS_DATE`; the cell below asserts it equals the cache header's `access=` field, and a rebuild stamps its own date into both and into every figure citation) | registry; the cache header |
| Provenance | Derivation record | `data/member_winters.csv`, written by the cell below with the parameters in its header; the cell that reads it **asserts** the header's SEASON, WINDOW, FRONT_RANGE, NINO34 and AUTUMN_MONTHS equal the parameter cell and its `access=` date equals `ACCESS_DATE` | this notebook |
| Provenance | Environment | `environment.yml` (beside this notebook; conda-forge, pinned); the versions this run resolved are printed under Imports | Module 2's environment file (Module 3's GPU step adds `environment-gpu.yml`) |

**Design defaults recorded here**: seven recorded design decisions, each with its reason and alternative. Each is a parameter in the cell above or a rule in the code; none is written into any answer; changing one is a refinement trial you record. 

| Default | Where | Why · the alternative |
|---|---|---|
| Forced response removed as a degree-`FORCED_DEG` polynomial in year fitted to the **training** members' ensemble mean | 2b | removes the shared warming path without using any test member · alternative: a per-member linear detrend |
| `wet` defined against one pooled training-member threshold, not a per-member quantile | 2b | a per-member quantile fixes every member's base rate, so Exercise 4 could never fail · alternative: that quantile |
| Test members drawn per half (`TEST_FRACTION`), named before any statistic is fit | 2b | keeps both halves on both sides of the line · alternative: one draw over both halves |
| `N_FOLDS = min(N_FOLDS_MAX, training members)`; repeats collapse to 1 when every fold holds one member | 2c | a small rebuilt cache can leave fewer members than folds; leave-one-member-out re-draws are identical · alternative: a fixed fold count |
| Rung 0 is three rules: always-dry, always-wet, Niño-sign; the register's first incumbent is the Niño-sign rule | 3a, 3f | the sign rule is the physical prior every rung must beat · alternative: always-wet as first incumbent, the sign rule a trial against it (Entry 6 shows where its precision meets that bar) |
| Promotion: one metric up measurably (corrected `p < 0.05`, gap > 0) with no mean loss in the other; each `p` uncorrected, the Bonferroni line printed | 3f | recall alone is gamed by predicting wet always; F1 hands always-wet about 0.5 · alternative: Holm or Bonferroni inside `verdict()`, or Module 3's guard |
| The 3d table carries fold-mean and pooled precision/recall, both labelled | 3d | they are different estimators; Exercise 7 computes the pooled one · alternative: one estimator only |

:::{note}
**Exercise 3 — the Metadata Document.** Check every cell the printed output can show, mark each *checked* or *not printed this run*, copy the table to the Handbook with one bias sentence per *Quality* row. → **Handbook entry 3**
:::


#### Deriving the member-winter table (one-time; cached)

Each Zarr chunk holds a whole global field, so deriving two box means for 100 members reads roughly 160 GB (the `chunking` line above computes it from the store: chunks × members × variables). The derivation runs **once**, writes
`data/member_winters.csv`, and every later cell reads the cache; `REBUILD = True` re-derives it with the parameters above — the record of how
the table was made is the code itself. The cell that reads the cache **asserts that its header was written with the
current `SEASON`, `WINDOW`, `FRONT_RANGE`, `NINO34` and `AUTUMN_MONTHS`**, and that its `access=` date is the parameter cell's `ACCESS_DATE`: a decision the cache silently ignored would be a fit that cannot be rerun. A rebuild is a new access — the header's `access=` field and every figure's citation take the day the rebuild read S3, not the committed date.
 


In [ ]:
CACHE = Path("data") / "member_winters.csv"       # relative to the notebook; no local absolute paths anywhere
REBUILD = False
N_MEMBERS_PER_HALF = 50                           # members per half when REBUILD = True: 50 = the full ensemble (a background job); fewer = a quick sample

BUCKET = "s3://ncar-cesm2-lens/atm/monthly"

def open_var(var, variant):
    return xr.open_zarr(f"{BUCKET}/cesm2LE-historical-{variant}-{var}.zarr",
                        consolidated=True, storage_options={"anon": True})[var]

def box_mean(da, box):
    sub = da.sel(**box)
    return sub.weighted(np.cos(np.deg2rad(sub.lat))).mean(("lat", "lon"))

def derive_member_winters(variant, members):
    """One row per member-winter: autumn Niño 3.4 (K, raw box mean) and the season-mean Front Range precipitation (mm/day)."""
    ts = open_var("TS", variant).sel(member_id=members)
    pr = (open_var("PRECC", variant) + open_var("PRECL", variant)).sel(member_id=members) * 8.64e7
    nino = box_mean(ts, NINO34).load()
    prfr = box_mean(pr, FRONT_RANGE).load()
    yr, mo = nino["time"].dt.year.values, nino["time"].dt.month.values
    rows = []
    for m in members:
        n, p = nino.sel(member_id=m).values, prfr.sel(member_id=m).values
        for y in range(WINDOW[0], WINDOW[1] + 1):
            son = n[(yr == y - 1) & np.isin(mo, AUTUMN_MONTHS)]
            if SEASON == "DJF":
                win = p[((yr == y - 1) & (mo == 12)) | ((yr == y) & np.isin(mo, [1, 2]))]
            else:  # "FMA"
                win = p[(yr == y) & np.isin(mo, [2, 3, 4])]
            if len(son) == 3 and len(win) == 3:
                rows.append(dict(member_id=m, forcing_variant=variant, winter=y,
                                 nino34_son_K=float(son.mean()), prect_winter_mm_day=float(win.mean())))
    return pd.DataFrame(rows)

# The header records the parameters the table was derived with; the same string is what the reader checks against.
PARAMS_RECORD = f"SEASON={SEASON} WINDOW={WINDOW} FRONT_RANGE={FRONT_RANGE} NINO34={NINO34} AUTUMN_MONTHS={AUTUMN_MONTHS}"
LEGACY_AUTUMN = "AUTUMN_MONTHS=[9, 10, 11]"   # a header written before AUTUMN_MONTHS was recorded (the committed cache, 2026-10-20) was derived with SON,
                                              #   the only autumn the derivation code had; it is appended for the comparison, so a changed AUTUMN_MONTHS still fails

def cache_matches_parameters(header_line):
    """Compare the parameters recorded in the cache header with the parameter cell; raise a readable error if they differ."""
    m = re.search(r"(SEASON=.*?)\s+n_members_per_half=", header_line)
    recorded = m.group(1) if m else "(no parameter record found in the header)"
    if m and "AUTUMN_MONTHS=" not in recorded:
        recorded += " " + LEGACY_AUTUMN
    if recorded != PARAMS_RECORD:
        raise AssertionError(
            "data/member_winters.csv was derived with different parameters than this notebook now declares.\n"
            f"  cache header : {recorded}\n  parameter cell: {PARAMS_RECORD}\n"
            "  Set REBUILD = True to re-derive the cache with the parameters you recorded (a background job for the full ensemble).")

def cache_access_date(header_line):
    m = re.search(r"\baccess=(\d{4}-\d{2}-\d{2})", header_line)
    return m.group(1) if m else "(no access= field in the header)"

if REBUILD or not CACHE.exists():
    from datetime import date
    ACCESS_DATE = date.today().isoformat()                      # a rebuild is a new access: the header and every figure citation carry THIS day
    LENS_CITATION = LENS_TEMPLATE.replace("ACCESS_DATE", ACCESS_DATE)
    print(f"deriving the table: access date {ACCESS_DATE} stamped into the header and the figure citations")
    CACHE.parent.mkdir(exist_ok=True)
    parts = []
    for variant in ["cmip6", "smbb"]:
        members = list(open_var("TS", variant).member_id.values[:N_MEMBERS_PER_HALF])
        parts.append(derive_member_winters(variant, members))
    table = pd.concat(parts, ignore_index=True)
    header = (f"# derived by 02_applications.ipynb: {PARAMS_RECORD} n_members_per_half={N_MEMBERS_PER_HALF} "
              f"RNG_SEED={RNG_SEED} access={ACCESS_DATE}\n")
    with open(CACHE, "w") as f:
        f.write(header); table.to_csv(f, index=False)

with open(CACHE) as f:
    cache_header = f.readline().strip()
cache_matches_parameters(cache_header)          # Exercise 1's SEASON (and WINDOW, the boxes, the autumn months) cannot be silently ignored
assert cache_access_date(cache_header) == ACCESS_DATE, (
    f"the cache header says access={cache_access_date(cache_header)} but the parameter cell says ACCESS_DATE = {ACCESS_DATE!r}: the figure "
    "citations must name the day the data were read — set ACCESS_DATE to the header's date, or set REBUILD = True for a new access")
table = pd.read_csv(CACHE, comment="#")

print(cache_header)
print("cache header matches the parameter cell:", PARAMS_RECORD, f"· access={ACCESS_DATE}")
header_members = 2 * int(re.search(r"n_members_per_half=(\d+)", cache_header).group(1))   # what the header says it holds
n_store = 2 * probe.member_id.size if probe is not None else None     # two halves, each the size of the store probed in 2a; unknown offline
of_store = (f"of {n_store} in the two stores ({'A SAMPLE' if table.member_id.nunique() < n_store else 'the full ensemble'})" if n_store else
            f"(offline: the stores' member count was not read; the header says it was derived from {header_members} members)")
print(f"{len(table)} member-winters · {table.member_id.nunique()} members {of_store} · winters {table.winter.min()}–{table.winter.max()}")
table.head()

### 2b Explore the target and the inputs

Three things are fit here, in an order that matters. **First the test members are named** — `TEST_FRACTION` of each half's members, drawn by
member from a `RandomState` the cell creates from `RNG_SEED` right before the draw (re-running the cell redraws the same members) — and nothing
below is fit on them. **Then the forced response is removed.** Every member follows the same forcing path by calendar year, so the raw SON Niño 3.4 box mean and
the winter precipitation both carry a forced trend (printed below as their correlation with year; Rodgers et al. 2021 §3.2). A smooth fit to the *training-member* ensemble mean is subtracted from every member alike, leaving the member's
own departure from the shared path — the ENSO state, not the decade. **Then the target is defined** against one threshold: `wet` = 1
when the detrended precipitation sits above the `WET_QUANTILE` of the pooled *training* members. Curve and threshold are transforms fit on
training members and applied to test members — as a `StandardScaler` inside a `Pipeline` is
(scikit-learn's *Common pitfalls and recommended practices*, §12.2.1) — written as `fit_transforms` and `apply_transforms` so that every validation fold
refits them. **Then the removal is checked**, with a line that can fail: the training members' mean anomaly per decade, in standard errors,
against a stated tolerance.  

Read the **base rate** first: with `WET_QUANTILE = 2/3` it is about one third, so accuracy of 0.67 is what "always dry" scores — why this
module reports precision and recall against the base rate (scikit-learn's *Classification metrics* guide). 


In [ ]:
# (1) Name the test members — per half, by member, from the declared seed. Nothing below is fit on them; 3g scores them once.
rng = np.random.RandomState(RNG_SEED)   # created HERE from the seed, so the draw is a function of RNG_SEED alone and re-running this cell in a live kernel
                                        # redraws the same members (a RandomState handed to GroupShuffleSplit advances on every call; a fresh one cannot drift)
test_members = []
for variant, half in table.groupby("forcing_variant"):
    hold = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=rng)
    _, held = next(hold.split(half, groups=half["member_id"]))
    test_members += sorted(half.iloc[held]["member_id"].unique())
table["split"] = np.where(table["member_id"].isin(test_members), "test", "train")
is_train = table["split"].eq("train")
print(f"test members (untouched until 3g): {test_members}")
print(f"training members: {sorted(table.loc[is_train, 'member_id'].unique())}")

# (2) + (3) The two transforms, written as functions so that 2c and 3c can refit them inside every validation fold.
years = np.arange(WINDOW[0], WINDOW[1] + 1)
ANOMALIES = {"nino34_anom": "nino34_son_K", "prect_anom": "prect_winter_mm_day"}   # anomaly column <- raw column

def fit_transforms(rows):
    """Fit on `rows` ONLY (all training members here; a fold's training members in 2c/3c):
    the forced curve of each series — a degree-FORCED_DEG polynomial in year fitted to the ensemble mean of `rows` — and
    the wet threshold — the WET_QUANTILE of the pooled detrended precipitation of `rows`."""
    curves = {}
    for anom, raw in ANOMALIES.items():
        ens_mean = rows.groupby("winter")[raw].mean().reindex(years)
        curves[anom] = pd.Series(np.polyval(np.polyfit(years, ens_mean.values, FORCED_DEG), years), index=years)
    prect_anom = pd.Series(rows["prect_winter_mm_day"].values - curves["prect_anom"].reindex(rows["winter"]).values)
    return dict(curves=curves, wet_threshold=float(prect_anom.quantile(WET_QUANTILE)))

def apply_transforms(rows, tf):
    """Subtract the fitted curves and label `wet` against the fitted threshold, on any rows (held-out ones included). Fits nothing."""
    out = rows.copy()
    for anom, raw in ANOMALIES.items():
        out[anom] = out[raw].values - tf["curves"][anom].reindex(out["winter"]).values
    out["wet"] = (out["prect_anom"] > tf["wet_threshold"]).astype(int)
    return out

TRANSFORMS = fit_transforms(table[is_train])   # fit on ALL training members: exploration (2b) and the one test (3g) use these
table = apply_transforms(table, TRANSFORMS)
forced, WET_THRESHOLD = TRANSFORMS["curves"], TRANSFORMS["wet_threshold"]
print(f"wet threshold (pooled training members) = {WET_THRESHOLD:+.4f} mm/day of detrended {SEASON} precipitation")


In [ ]:
# Did removing the forced response work?
# (a) The forced trend the removal has to take out: corr(raw series, year) over the training members.
# (b) THE CHECK. A forced path the curve missed (too low a degree, a mid-century bump) survives the subtraction as decades whose mean
#     anomaly sits away from zero. Each decade's mean anomaly over the training members is divided by its standard error; the removal
#     PASSES only if every decade of both series lies within ±DECADE_TOL standard errors. (corr(anomaly, year) over the training members
#     is not used: it is zero by construction for any FORCED_DEG >= 1, so it could never fail.) Try FORCED_DEG = 0 and watch it fail.
from scipy.stats import norm
DECADE_TOL = 3.0
d = table[is_train]
decade = (d["winter"] // 10) * 10
for anom, raw in ANOMALIES.items():
    print(f"corr({raw:20s}, year) = {d[raw].corr(d['winter']):+.3f}   (the forced trend in the raw series, training members)")
removal_ok, n_tests = True, 0
for anom in ANOMALIES:
    g = d.groupby(decade)[anom]
    z = g.mean() / (d[anom].std() / np.sqrt(g.size()))
    worst = z.abs().idxmax(); ok = bool((z.abs() <= DECADE_TOL).all())
    removal_ok &= ok; n_tests += len(z)
    print(f"{anom:12s} decade-mean anomaly in standard errors: worst {z[worst]:+.2f} (the {worst}s) · tolerance ±{DECADE_TOL:.0f} → {'PASS' if ok else 'FAIL'}")
false_alarm = 1 - (1 - 2 * norm.sf(DECADE_TOL)) ** n_tests
print(f"forced-response removal: {'PASS' if removal_ok else 'FAIL — a forced path survived the curve; change FORCED_DEG and re-read 2b'}"
      f"   ({n_tests} decade means; chance a correct removal trips the tolerance somewhere ≈ {false_alarm:.0%})")


In [ ]:
train_tbl = table[is_train]
base_rate = train_tbl["wet"].mean()
print(f"base rate P(wet), training members = {base_rate:.3f}  (n = {len(train_tbl)} member-winters)")
print("\nper-half base rate (training members) — the Exercise 4 check; free to differ because the threshold is one for both halves:")
print(train_tbl.groupby("forcing_variant")["wet"].agg(["mean", "size"]).round(3))
print("\nper-half detrended precipitation (training members):")
print(train_tbl.groupby("forcing_variant")["prect_anom"].agg(["mean", "std", "median"]).round(3))

def corr_in(d):
    return d["nino34_anom"].corr(d["prect_anom"])

per_member_corr = train_tbl.groupby("member_id").apply(corr_in)
print("\nper-member correlation(nino34_anom, prect_anom), training members:")
print(per_member_corr.round(3).to_string())
# The two lines Entry 4 pastes: do the members agree on a sign (counted, not eyeballed), and do the two halves agree?
pos, neg = int((per_member_corr > 0).sum()), int((per_member_corr < 0).sum())
print(f"\nper-member sign count: {pos} positive · {neg} negative of {len(per_member_corr)} training members · "
      f"mean {per_member_corr.mean():+.3f} · SD {per_member_corr.std(ddof=1):.3f}")
half_means = train_tbl.groupby(["forcing_variant", "member_id"]).apply(corr_in).groupby(level=0).mean()
print("per-half mean correlation: " + " · ".join(f"{half} {v:+.3f}" for half, v in half_means.items()))


In [ ]:
d = table[is_train]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(d["prect_anom"], bins=30, color="#C9A23A", edgecolor="#C9A23A", linewidth=0.5)
axes[0].axvline(WET_THRESHOLD, lw=1, c="k", ls="--", label="wet threshold (training)"); axes[0].legend()
axes[0].set(xlabel=f"{SEASON} Front Range precipitation, forced response removed (mm/day)", ylabel="member-winters", title="Target distribution (training members)")
for wet, (c, m, lab) in {0: ("#1b9e77", "o", "dry"), 1: ("#e7298a", "^", "wet")}.items():   # colour AND marker shape: hatching is invisible at this marker size, and colour alone fails colour-blind readers
    dd = d[d.wet == wet]
    axes[1].scatter(dd["nino34_anom"], dd["prect_anom"], s=12, alpha=.6, c=c, marker=m, label=lab)
axes[1].axvline(0, lw=.8, c="k"); axes[1].legend()
axes[1].set(xlabel="autumn Niño 3.4 anomaly, forced response removed (K)", ylabel=f"{SEASON} precipitation anomaly (mm/day)", title="Input vs target")
cite(fig); plt.tight_layout()


:::{note}
**Exercise 4 — the two halves.** From the per-half tables printed before the figure: is pooling cmip6 and smbb justified, and did the sign you wrote in Entry 1 appear? → **Handbook entry 4**
:::

**Reading the scatter.** NWS Boulder's sign for east of the Divide is positive — warmer autumn Pacific, wetter winter — and the vertical line at zero
splits the cloud into the winters the sign rule calls wet (right) and dry (left); if the magenta triangles (wet) sit to the right more often
than the teal circles (dry), the sign is the one you wrote in Exercise 1. Then read the per-member
correlation lines printed above against Stagge et al.'s poles (their northern model "explained 67.1% of winter precipitation", the southern
"less variance (34.5%)"; this box sits between them). Answer from the printed lines, not from this text: the `per-member sign count` line says whether the
members share a sign, and the `per-half mean correlation` line whether the two halves agree and whether either is far from zero against the SD beside the count. Is the scatter
a slope or a cloud?
 


### 2c Split by ensemble member

This is the step the course changes from Unidata's notebooks (they split "randomly"), and the one O2's *split* failure names: *training and
scoring sets that share ensemble members, so the "unseen" score is inflated.* scikit-learn's guidance on grouped
cross-validation (BSD-3-Clause): "all the samples in the validation fold come from groups that are not represented at all in the paired training fold." 

Here the group is the **member**, for three reasons the code below makes measurable:

1. **The question is about members never seen.** Part 1 asks what the autumn state says about *a* member's coming winter, so the score has to
   be on a realisation the model has not met; a member-blind split scores new *years* of members it already knows.
2. **The normalisation constants.** The forced curve and the wet threshold are fit on training members only (2b) and refit inside every
   validation fold (`score_splits` calls `fit_transforms` per fold). Fit before the split, the held-out winters would help fit the curve and the
   threshold that label them — the same leak as a `StandardScaler` fit on the whole table; Exercise 5 asks what that score would measure.
3. **Dependence within and between members is measured, not assumed.** Rodgers et al. 2021 §2.2 is why `WINDOW`
   starts late — initial-condition memory fades over decades — but Rodgers give no figure for how much is left after fifty years, so the cell
   prints the mean correlation of the detrended series between micro-siblings (the same start year, as the CESM2 LE project page defines them) and between
   unrelated training members, with a line that **fails if siblings share more**. Siblings are not grouped in the split; that line is what
   licenses it, and Entry 5 asks you to paste it. The same cell prints the lag-1 autocorrelation inside members, and the contrast cell scores one
   model three ways on the same members (the table below): with one input that carries no member identity, whether either leak bites here is a
   number, not a claim.  

| Arm (the contrast cell below) | Split unit | Forced curve and wet threshold fit on | What a held-out score measures |
|---|---|---|---|
| A — this notebook's pipeline | member | that fold's training members only (refit in every fold) | new members |
| B — transforms fit before the split | member | all the training members, before the folds (the test members still outside) | new members scored with transforms they helped fit — Exercise 5's "tidier" pipeline at fold level |
| C — member-blind split | winter (`StratifiedKFold`, members ignored) | that fold's training winters only | new years of members the model already knows |

Two splits are made, both by member, both from the declared seed: the **test members** (named in 2b, touched once in 3g) and **member-grouped
folds** over the rest, `N_FOLDS = min(N_FOLDS_MAX, training members)`. The folds are stratified on the labels fit on all the training members;
the labels scored inside each fold are refit on that fold's training members, so the all-training labels decide only which members share a fold.
The contrast's model is boosting at the base-rate decision threshold (3e), so that it *calls wet*: the cell's first printed line scores the same model at scikit-learn's default 0.5 and shows how rarely it calls wet there — a model that almost never calls wet has a precision resting on a handful of calls, and a gap in it is a non-test — then the cell prints each arm's wet-call count and says so.
 


In [ ]:
FEATURES = ["nino34_anom"]           # the one input; a new input is a costed 3h next change (3b), not a 3f trial in this notebook
train, test = table[is_train].reset_index(drop=True), table[~is_train].reset_index(drop=True)   # raw columns kept: folds refit the transforms

# No member on both sides of the line.
shared = set(train.member_id) & set(test.member_id)
assert not shared, f"split leaks members: {shared}"

n_train_members = train.member_id.nunique()
N_FOLDS = min(N_FOLDS_MAX, n_train_members)      # the fold count follows the data
assert n_train_members >= 2, (f"only {n_train_members} training member(s): member-grouped folds need at least 2 — "
                              f"rebuild the cache with a larger N_MEMBERS_PER_HALF (REBUILD = True)")
print(f"train: {n_train_members} members / {len(train)} winters   test: {test.member_id.nunique()} members / {len(test)} winters")
print(f"base rate train {train.wet.mean():.3f}   (the test members' base rate is printed once, in 3g; nothing here reads their labels)")
print(f"N_FOLDS = min(N_FOLDS_MAX={N_FOLDS_MAX}, {n_train_members} training members) = {N_FOLDS}")

def member_folds(seed):
    """Member-grouped, class-balanced folds over the TRAINING members only. The balancing reads the labels fit on all training members
    (2b); the labels scored inside a fold are refit on that fold's training members by score_splits, so this decides fold membership only."""
    return StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

X_tr, y_tr, g_tr = train[FEATURES].values, train["wet"].values, train["member_id"].values
for k, (a, b) in enumerate(member_folds(RNG_SEED).split(X_tr, y_tr, g_tr)):
    assert not set(g_tr[a]) & set(g_tr[b]), f"fold {k} leaks members"
print(f"{N_FOLDS} member-grouped folds verified: no member appears in both the fitting and the scoring side of any fold")

# When every fold holds exactly one member, every re-draw is the same partition: repeating it would count the same per-fold
# differences N_REPEATS times over and inflate the paired test in 3f. The repeats then collapse to 1 (the full table restores them).
N_REPEATS_EFFECTIVE = N_REPEATS if n_train_members > N_FOLDS else 1
if N_REPEATS_EFFECTIVE == 1:
    print(f"NOTE: {n_train_members} training members and {N_FOLDS} folds is leave-one-member-out; fold re-draws are identical, "
          f"so 3f uses N_REPEATS_EFFECTIVE = 1 instead of {N_REPEATS}")

# Within-member dependence, measured: lag-1 autocorrelation of the input and the target inside each training member.
def lag1(s):
    v = np.asarray(s, dtype=float)
    return float(np.corrcoef(v[:-1], v[1:])[0, 1])

dependence = train.groupby("member_id").agg(lag1_nino34_anom=("nino34_anom", lag1), lag1_wet=("wet", lag1)).round(3)
print("\nlag-1 autocorrelation within training members (what consecutive winters of one member share):")
print(dependence.to_string()); print("mean:", dependence.mean().round(3).to_dict())

# Between-member dependence, measured: the correlation over the window between two training members' detrended series, averaged over micro-sibling
# pairs (the same start year — the i1231/i1251/i1281/i1301 token in member_id, as the CESM2 LE project page defines them) and over every other pair. Had the
# initial-condition memory of Rodgers §2.2 survived to WINDOW[0], siblings would correlate more than unrelated pairs; the last line FAILS if they do.
MICRO_STARTS = {"1231", "1251", "1281", "1301"}
start_year = {m: re.search(r"i(\d{4})p", m).group(1) for m in train.member_id.unique()}
print(f"\ncorrelation between training members over {WINDOW[0]}–{WINDOW[1]} (micro-siblings share a start year; the memory check):")
for anom in ANOMALIES:
    wide = train.pivot(index="winter", columns="member_id", values=anom); corr = wide.corr()
    sib, unrel = [], []
    for a, b in itertools.combinations(wide.columns, 2):
        (sib if start_year[a] == start_year[b] and start_year[a] in MICRO_STARTS else unrel).append(corr.loc[a, b])
    sib, unrel = np.array(sib), np.array(unrel)
    if len(sib) < 2 or len(unrel) < 2:
        print(f"{anom:12s} too few member pairs of one kind to compare ({len(sib)} sibling, {len(unrel)} unrelated): rebuild with more members"); continue
    excess = sib.mean() - unrel.mean(); se = np.sqrt(sib.var(ddof=1) / len(sib) + unrel.var(ddof=1) / len(unrel))
    print(f"{anom:12s} micro-siblings {sib.mean():+.3f} ({len(sib)} pairs, SD {sib.std(ddof=1):.3f}) · unrelated {unrel.mean():+.3f} ({len(unrel)} pairs, SD {unrel.std(ddof=1):.3f}) "
          f"· siblings − unrelated {excess:+.3f} (±{se:.3f}, a lower bound: the pairs share members, so they are not independent draws) → "
          + ("PASS: no sibling excess detectable (|excess| ≤ 2 SE, SE a lower bound) — the member is taken as the unit" if abs(excess) <= 2 * se else
             "FAIL: siblings still move together — group the folds by start year, or start WINDOW later"))


In [ ]:
# The measured contrast (Exercise 5). One function scores every validation run in this notebook: score_splits.
def score_splits(model, rows, splits, refit_transforms=True):
    """Fit on each split's training rows, score its held-out rows. With refit_transforms=True (this notebook's pipeline) the forced
    curves and the wet threshold are refit on the split's training rows only (fit_transforms, 2b) and applied to both sides, so a
    held-out row is centred and labelled by transforms it did not help fit. With False the columns already in `rows` are used: the
    transforms fit once on ALL training members, before the folds — so a held-out fold member helped fit them; the test members are still outside. Returns per-fold precision, recall, F1, accuracy and AP for class 1
    (wet), plus the summed confusion matrix (rows = true, columns = predicted)."""
    out = dict(precision=[], recall=[], f1=[], accuracy=[], ap=[]); cms = []
    for a, b in splits:
        fit_rows, held_rows = rows.iloc[a], rows.iloc[b]
        if refit_transforms:
            tf = fit_transforms(fit_rows)
            fit_rows, held_rows = apply_transforms(fit_rows, tf), apply_transforms(held_rows, tf)
        Xa, ya, Xb, yb = fit_rows[FEATURES].values, fit_rows["wet"].values, held_rows[FEATURES].values, held_rows["wet"].values
        model.fit(Xa, ya); yhat = model.predict(Xb)
        score = model.predict_proba(Xb)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(Xb)
        out["precision"].append(precision_score(yb, yhat, zero_division=0)); out["recall"].append(recall_score(yb, yhat, zero_division=0))
        out["f1"].append(f1_score(yb, yhat, zero_division=0)); out["accuracy"].append(accuracy_score(yb, yhat))
        out["ap"].append(average_precision_score(yb, score)); cms.append(confusion_matrix(yb, yhat, labels=[0, 1]))
    return {k: np.array(v) for k, v in out.items()}, np.sum(cms, axis=0)

# Three arms, the same model (the ladder's most flexible rung, settings from the parameter cell) at the base-rate DECISION THRESHOLD (3e),
# so that it calls wet. The first printed line scores arm A's model at sklearn's default 0.5 (one fold draw) and prints its wet-call count and
# recall — measured here, never typed in: the precision of a model that almost never calls wet rests on a handful of calls, and its precision
# and recall gaps are a non-test. The threshold moves the operating point, not the ranking (AP), so the leak is measured on numbers that can
# move. N_REPEATS_EFFECTIVE fold draws per arm.
# Arm A is this notebook's pipeline; B differs from A ONLY in where the transforms are fit; C differs from A ONLY in the split unit.
# Each arm prints its scores with their per-fold SDs and its wet-call count, then its gaps against A with the SD of the per-fold gaps — a rough
# scale here; 3f's paired test is the measurement — and whether its precision/recall gaps are readable at all.
CONTRAST_THRESHOLD = 1 - WET_QUANTILE          # the designed base rate, the same threshold the example trial in 3f records
MIN_WET_CALL_SHARE = 0.01                      # below one wet call per hundred scored winters, precision and recall gaps are a NON-TEST
contrast_model = lambda seed: FixedThresholdClassifier(GradientBoostingClassifier(**BOOSTING_SETTINGS, random_state=seed),
                                                       threshold=CONTRAST_THRESHOLD, response_method="predict_proba")
arms = {"A member split, transforms refit in each fold   ": dict(runs=[], cm=np.zeros((2, 2), dtype=int)),
        "B member split, transforms fit before the split ": dict(runs=[], cm=np.zeros((2, 2), dtype=int)),
        "C member-blind split, transforms refit in fold  ": dict(runs=[], cm=np.zeros((2, 2), dtype=int))}
# The reference line: the same model at sklearn's default 0.5, arm A's split (one fold draw) — how rarely it calls wet is printed, not asserted.
_s05, _cm05 = score_splits(GradientBoostingClassifier(**BOOSTING_SETTINGS, random_state=RNG_SEED), train,
                           list(member_folds(RNG_SEED).split(X_tr, y_tr, g_tr)), refit_transforms=True)
print(f"for reference — the same model at sklearn's default threshold 0.5 (arm A's split, one fold draw): wet calls {int(_cm05[:, 1].sum())} of "
      f"{int(_cm05.sum())} scored winters ({_cm05[:, 1].sum() / _cm05.sum():.2%}), recall {_s05['recall'].mean():.3f} — too few calls to read a gap in")
for rep in range(N_REPEATS_EFFECTIVE):
    seed = RNG_SEED + rep
    by_member = list(member_folds(seed).split(X_tr, y_tr, g_tr))
    by_year = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(X_tr, y_tr))   # member-blind: winters shuffled
    for (label, store), (splits, refit) in zip(arms.items(), [(by_member, True), (by_member, False), (by_year, True)]):
        s, cm = score_splits(contrast_model(seed), train, splits, refit_transforms=refit)
        store["runs"].append(s); store["cm"] += cm
CONTRAST_METRICS = ["ap", "precision", "recall", "accuracy"]
pooled_arm = {label: {m: np.concatenate([s[m] for s in st["runs"]]) for m in CONTRAST_METRICS} for label, st in arms.items()}
ref = pooled_arm[next(iter(pooled_arm))]
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
print(f"boosting at threshold {CONTRAST_THRESHOLD:.3f} (the base rate; 0.5 is sklearn's default), {N_REPEATS_EFFECTIVE} fold draw(s) × {N_FOLDS} folds per arm. "
      f"Per arm: scores (mean ± per-fold SD) and wet calls; then gap = arm − A (mean ± SD of the per-fold gaps)")
for label, s in pooled_arm.items():
    cm = arms[label]["cm"]; wet_calls, n_scored = int(cm[:, 1].sum()), int(cm.sum()); share = wet_calls / n_scored
    readable = share >= MIN_WET_CALL_SHARE
    print(f"{label} AP {s['ap'].mean():.3f}   precision {s['precision'].mean():.3f} ± {sd(s['precision']):.3f}   recall {s['recall'].mean():.3f} ± {sd(s['recall']):.3f}   "
          f"accuracy {s['accuracy'].mean():.3f}   wet calls {wet_calls} of {n_scored} scored winters ({share:.1%})")
    gaps = {m: s[m] - ref[m] for m in ["ap", "precision", "recall"]}
    print(f"{'':{len(label)}} gap vs A: AP {gaps['ap'].mean():+.4f} ± {sd(gaps['ap']):.4f}   precision {gaps['precision'].mean():+.4f} ± {sd(gaps['precision']):.4f}   "
          f"recall {gaps['recall'].mean():+.4f} ± {sd(gaps['recall']):.4f}   → "
          + (f"gaps readable (wet calls ≥ {MIN_WET_CALL_SHARE:.0%} of winters)" if readable else
             f"precision/recall gaps a NON-TEST: wet calls below {MIN_WET_CALL_SHARE:.0%} of winters; read AP only"))


:::{note}
**Exercise 5 — the leak you cannot see in a split.** A colleague's "tidier" pipeline fits the transforms on the whole table before any split: say what 3g would then measure, read the contrast block (two lines per arm: scores with wet-call counts, then gaps with their SDs and a readable/NON-TEST verdict) and the two dependence lines above, and paste the assertion, lag-1, sibling and contrast lines. → **Handbook entry 5**
:::


## Part 3: Model development

### 3a Choose the algorithm — the ladder

Unidata compares two algorithms; this module climbs a **ladder of three**
(each from scikit-learn, BSD-3-Clause): 

| Rung | Estimator | What you can read from it | What it costs |
|---|---|---|---|
| 0 | always-dry · always-wet — the two bars (accuracy; precision at the base rate) · Niño-sign rule — the physical prior and 3f's first incumbent, no bar | nothing to fit | nothing — two rules set the bars and one is the prior every rung must beat |
| 1 | `LogisticRegression` | a coefficient per input, in log-odds | assumes a linear boundary |
| 2 | `DecisionTreeClassifier(max_depth=…)` | the rules themselves | "can be unstable"; overfits without a depth or leaf limit |
| 3 | `GradientBoostingClassifier` | little directly; "an excellent model … for tabular data" | run time; readability; more settings to record |

The dataset is small, so the guide's own note points at `GradientBoostingClassifier` rather than the histogram variant; each rung is compared
with the incumbent by the paired test in 3f; only a rung that wins becomes the incumbent, and the next rung is compared with that. 

**Why each rung is on the ladder.** Rung 0 because the physical prior is weak: two rules that fit nothing set the bars (always-dry on accuracy; always-wet on precision, at
the base rate) and one, the Niño-sign rule, is the prior every rung must beat — no bar; if nothing clears the bars the ladder has still
answered the question. Rung 1 because one coefficient in log-odds is the first thing a scientist wants to read. Rung 2
because its rules are readable — does the signal live only in strong events, as NWS Boulder's "moderate to strong" suggests? — under the depth cap
`TREE_SETTINGS` sets, because trees "can be unstable". Rung 3 as the most flexible model a one-input table can justify; it is also the model the
2c contrast cell uses — at the base-rate threshold, so that it calls wet often enough for a gap to be read (2c's first printed line shows how rarely it calls wet at 0.5) — because a flexible model is the one that can memorise a member. 


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
class _SignRule(BaseEstimator, ClassifierMixin):
    """Rung 0c: wet if the autumn Niño 3.4 anomaly (feature 0) is positive. Nothing is fit; its only score is its 0/1 decision."""
    def fit(self, X, y=None): self.classes_ = np.array([0, 1]); return self
    def predict(self, X): return (np.asarray(X)[:, 0] > 0).astype(int)
    def predict_proba(self, X): p = self.predict(X); return np.c_[1 - p, p]

def ladder(seed):
    """The candidates, each a Pipeline so preprocessing is fit on the training fold only (sklearn §12.2.1).
    Rung 0 is three rules: always-dry (the majority class — its wet precision and recall are 0 by construction, so it is a bar on
    accuracy only), always-wet (recall 1 at precision = the base rate — the bar precision is read against) and the Niño-sign rule."""
    return {
        "always-dry": DummyClassifier(strategy="most_frequent"),
        "always-wet": DummyClassifier(strategy="constant", constant=1),
        "nino-sign":  Pipeline([("rule", _SignRule())]),
        "logistic":   Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(random_state=seed))]),
        "tree":       DecisionTreeClassifier(**TREE_SETTINGS, random_state=seed),
        "boosting":   GradientBoostingClassifier(**BOOSTING_SETTINGS, random_state=seed),
    }
RUNG0 = ["always-dry", "always-wet", "nino-sign"]


### 3b Choose the inputs

In Unidata's 3b you choose the inputs; here Part 1 fixed one — the autumn Niño 3.4 anomaly — and this section records what was not chosen
and what each would cost, so the choice you make is in 3h, costed. One input was fixed because the question is about ENSO and a one-input model
is one a scientist can read whole. The catalogue offers five other monthly fields in both halves — PSL, TREFHT, Z3, U and T — each a fair input only as an **autumn**
(SON) value, known before the winter starts. 

| Autumn (SON) field, in both halves | What it stands for | In time before the winter? | In this notebook |
|---|---|---|---|
| TS over `NINO34` | the ENSO state | yes | the one input (`FEATURES`) |
| PSL or Z3 over the North Pacific | a stand-in for the autumn PNA state | yes | the strongest candidate for 3h |
| TREFHT over `FRONT_RANGE` | the box's own autumn temperature | yes, but it tells of the box's own state, not the teleconnection | a weaker candidate |
| U, T | zonal wind and temperature, three-dimensional fields | yes, as SON means | not named by the forecaster's deck |
| any field averaged over the winter months | the outcome's own season | no — known only after the outcome | never |
| `forcing_variant` | which half a member belongs to | — | not an input: Exercise 4 decides whether the halves pool |

None of them is a 3f trial here: the committed cache carries one input column, and a new autumn field means a new box mean in
`derive_member_winters` (2a) and a rebuild of the cache from S3 (roughly 160 GB; the course's one full derivation took 33 minutes on a fast link, 2026-10-20 — on home broadband expect an hour or more). So an input is a **costed next
change for 3h**, never appended to `FEATURES` here; NWS Pueblo's deck, which names PNA and NAO as the patterns with the "highest correlations
wet/dry", ranks the candidates. 

### 3c Train

Training guards against O2's *train* failure: a fit that cannot be rerun. Every estimator takes `random_state=seed` derived from the declared
`RNG_SEED`, every preprocessing step sits inside a `Pipeline` so it is fit on the training fold only, and every trial's settings are written to
the register in 3f. 


In [ ]:
def run_folds(model, rows, seed):
    """Member-grouped folds over the training members (2c), scored by score_splits with the transforms refit inside every fold."""
    return score_splits(model, rows, member_folds(seed).split(rows[FEATURES].values, rows["wet"].values, rows["member_id"].values))

seed = RNG_SEED
models = ladder(seed)
results = {name: run_folds(m, train, seed) for name, m in models.items()}
print("trained:", ", ".join(models))


### 3d Validate — confusion matrix, precision, recall

Validation guards against O2's *validate* failure: *tuning with no held-out score to show whether the model is memorising.* Every number here
comes from members the fit never saw: in every fold the model, the forced curve and the wet threshold are fit on that fold's training members
only, so a held-out member is centred and labelled by transforms it did not help fit. The confusion matrix follows scikit-learn's convention — **rows = true class,
columns = predicted** (scikit-learn's *Classification metrics* guide); precision reads down the "predicted wet" column, recall along the "actual wet" row.
Both are for the *wet* class, and both are read against the base rate. 

The table carries **two estimators of the same thing, labelled**. `precision_fold` / `recall_fold` are the mean (± SD) of the per-fold values,
with `zero_division=0`, so a fold in which a model never predicts wet contributes 0. `precision_pooled` / `recall_pooled` are read off the
**summed confusion matrix** — one division over all folds' counts — which is what the matrices show and what Exercise 7 computes by hand; the two
differ whenever folds differ in size or in how often the model predicted wet. Accuracy, F1 and average precision (AP) stand beside them because
precision and recall alone can be gamed by predicting one class (always-wet has recall 1 at precision = base rate); a constant or random score
has AP = base rate (the same guide). 


In [ ]:
def pooled(cm):
    """Precision and recall for wet from a summed confusion matrix (rows = true, columns = predicted): one division over all folds."""
    tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]
    return dict(precision_pooled=tp / (tp + fp) if tp + fp else 0.0, recall_pooled=tp / (tp + fn) if tp + fn else 0.0)

summary = pd.DataFrame({name: dict(precision_fold=s["precision"].mean(), precision_sd=s["precision"].std(ddof=1),
                                   recall_fold=s["recall"].mean(), recall_sd=s["recall"].std(ddof=1), **pooled(cm),
                                   called_wet=cm[:, 1].sum() / cm.sum(),   # the pooled share of winters called wet (03's column of the same name): how thin a rung's evidence is
                                   accuracy=s["accuracy"].mean(), f1=s["f1"].mean(), ap=s["ap"].mean())
                        for name, (s, cm) in results.items()}).T.round(3)
summary["base_rate"] = round(float(y_tr.mean()), 3)
print("nino-sign summed confusion matrix (rows = true, columns = predicted) — the four counts Exercise 7 uses; the figure below draws the same matrix:")
print(pd.DataFrame(results["nino-sign"][1], index=["true dry", "true wet"], columns=["pred dry", "pred wet"]))
summary


In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(3.0 * len(results), 3.4))
for ax, (name, (s, cm)) in zip(axes, results.items()):
    ConfusionMatrixDisplay(cm, display_labels=["dry", "wet"]).plot(ax=ax, colorbar=False, cmap=GOLD_CMAP)
    ax.set_xlim(-0.85, 1.85); ax.set_ylim(1.85, -0.85)   # a canvas frame around the matrix (the course's panel surface)
    pr = pooled(cm)
    ax.set_title(f"{name}\npooled P={pr['precision_pooled']:.2f} R={pr['recall_pooled']:.2f}", fontsize=9)
fig.suptitle(f"Summed confusion matrices over {N_FOLDS} member-grouped folds (rows = true, columns = predicted); titles = pooled precision/recall", fontsize=9)
cite(fig); plt.tight_layout()


:::{note}
**Exercise 6 — the bar.** From the 3d table, record the three rung-0 rows with the base rate; say which of the two base-rate rows bars which metric, read off the rows — the sign rule is recorded as the physical prior, no bar (its row asks where its precision sits against the always-wet bar). → **Handbook entry 6**
:::

:::{note}
**Exercise 7 — read the matrices.** From the `nino-sign` matrix printed above the table, compute precision and recall for *wet* by hand; then relabel the four counts as hits, false alarms, misses and correct rejections, name POD (= recall) and FAR (= 1 − precision), and report frequency bias where precision and recall disagree. MET, the Developmental Testbed Center's verification package (NSF NCAR, NOAA, CSU/CIRA and CU/CIRES), prints the same four counts with the axes swapped — rows = forecast, columns = observation, "Yes" first — so hits and correct rejections exchange corners (scikit-learn's bottom-right and top-left become MET's top-left and bottom-right) while false alarms stay top-right and misses bottom-left in both layouts — read every cell off the axis labels, not off scikit-learn's corners (the MET User's Guide, contingency-table section). → **Handbook entry 7** 
:::

### 3e Evaluate and refine

Read the table and the matrices as a scientist, not a scoreboard. The decision in Part 1 is a seasonal lean, and the two ways to be wrong cost
differently: a missed wet winter (the true-wet row's "predicted dry" cell; recall falls) leaves the outlook at equal chances — a cheap miss; a
false alarm (the true-dry row's "predicted wet" cell; precision falls) leans an outlook wet into a dry winter — the costlier error for an office
that will be held to it. The decision needs both numbers. At the default threshold a probabilistic
model says "wet" only when it is more than half sure, and with a weak signal it may rarely be: a rung whose recall sits near zero is at an
operating point, not a verdict — the AP column, which reads the ranking at every threshold, says whether a rung separates wet from dry at all. 3f runs **one example refinement trial** so you can see the pattern: it changes only logistic's decision threshold, from 0.5 to
the designed base rate (1 − `WET_QUANTILE`), and the cell below shows that trade-off on the validation folds first.
 


In [ ]:
# The decision threshold is a refinement trial, not a silent change. A probabilistic model predicts wet when P(wet) > 0.5 by default
# (sklearn's predict). When the base rate is one third and the signal is weak, P(wet) may rarely cross 0.5, and a rung then predicts
# dry almost always — check the 3d table's recall and accuracy columns. Lowering the threshold trades precision for recall: it moves the
# operating point along the same ranking, it does not add skill (AP does not move). This cell only SHOWS the trade-off on the
# validation folds. A threshold you decide to keep is recorded as a trial in 3f (FixedThresholdClassifier) and compared with the
# incumbent by the paired test like any other change — and the threshold you choose is part of the settings the register records.
thresholds = [0.5, 1 - WET_QUANTILE]       # sklearn's default, and the designed base rate (the measured one is printed in 2b)
rows = []
for thr in thresholds:
    m = FixedThresholdClassifier(ladder(RNG_SEED)["logistic"], threshold=thr, response_method="predict_proba")
    s, cm = run_folds(m, train, RNG_SEED)
    rows.append(dict(model="logistic", threshold=round(thr, 3), **pooled(cm), f1=s["f1"].mean(), ap=s["ap"].mean(),
                     predicted_wet=int(cm[:, 1].sum()), of=int(cm.sum())))
pd.DataFrame(rows).round(3)


### 3f Refinement trials — the paired test

Refinement guards against O2's *refine* failure: *stopping at the first configuration tried.* Unidata's instruction is to run trials "until … they no
longer improve" — judged against the **run-to-run spread**. Scores from the same folds are not independent, so a
plain t-test under-estimates the variance; the course uses the **Nadeau–Bengio corrected paired t-test**, one-sided (a candidate must beat the incumbent), from scikit-learn's own example,
*Statistical comparison of models using grid search* (BSD-3-Clause). **Which spread:** the corrected standard
error of the per-fold differences, which the `_t` column divides the gap by. The register also prints `_sd`, the plain per-fold standard
deviation, for scale — on this design the corrected standard error is about half that per-fold SD, and about 3.7 times the naive standard error a plain paired t would use (that SD divided by the square root of the 50 paired scores): inflating the naive error is the point of the correction, and the factor is in `corrected_std` below — and `_sd` is **not** the test. A gap inside the corrected error is **not a difference**. The example warns that "if
multiple models are statistically compared, a multiple comparisons correction is needed"; the register does **not** apply one — each printed `p`
is per test, uncorrected — so it **bounds the looks**: six tested rows, two metrics each. Under every register the
notebook prints the **Bonferroni line**, 0.05 ÷ the looks taken so far: a `p` under 0.05 but above that line is a promotion under the
uncorrected rule only (design default 6). 

**What a trial is scored on.** Recall alone rewards predicting wet more often (always-wet scores 1.0); precision alone rewards predicting it
almost never; AP cannot see the threshold 3e teaches you to trial. So a candidate is **promoted only if it improves one of the two numbers the
decision uses measurably — corrected `p < 0.05` with a positive gap — without giving the other back on average (mean gap ≥ 0)**; both tests sit
in every register row. The six tested rows are spent as: the ladder's three rungs and the example trial — four already spent when the notebook arrives — then up to two trials of your own.
 

**What this rule cannot see.** It compares a candidate with the *incumbent* only, and the first incumbent is the Niño-sign rule — the physical
prior, not a rule that cleared a bar; your Entry 6 says whether its precision sits above or below the always-wet bar. If *below*, a candidate
with no skill at all — precision merely *at* the base rate — can beat it on precision; the example trial can be exactly that, so read any
promotion it earns as a promotion over the sign rule, not as skill. The 3g interval lines catch it, reading the tested
model against the base rate of the same winters; Module 3's 3f adds the guard this rule lacks (AP above the incumbent *and* the base rate). 


In [ ]:
# Copyright (c) 2007-2026 The scikit-learn developers. BSD-3-Clause; full text in LICENSES/scikit-learn-BSD-3-Clause.txt. From examples/model_selection/plot_grid_search_stats.py
def corrected_std(differences, n_train, n_test):
    """Corrects standard deviation using Nadeau and Bengio's approach."""
    kr = len(differences)
    corrected_var = np.var(differences, ddof=1) * (1 / kr + n_test / n_train)
    return np.sqrt(corrected_var)

def compute_corrected_ttest(differences, df, n_train, n_test):
    """Computes right-tailed paired t-test with corrected variance."""
    mean = np.mean(differences)
    std = corrected_std(differences, n_train, n_test)
    if std == 0:                      # course addition: identical scores in every fold — no spread, no difference (guards 0/0)
        return 0.0, 1.0
    t_stat = mean / std
    p_val = t_dist.sf(t_stat, df)       # course change: t_stat, not np.abs(t_stat) — a right tail as the docstring says, so a LOSS prints p → 1;
    return t_stat, p_val                #   the example's abs printed a loss's left-tail probability under a right-tailed label

METRICS = ["recall", "precision"]      # the two numbers the decision uses; both are tested in every trial

def trial(make_model, rows):
    """Repeat the member-grouped folds N_REPEATS_EFFECTIVE times with different fold draws; per-fold precision and recall (kr values each)."""
    scores = {m: [] for m in METRICS}
    for rep in range(N_REPEATS_EFFECTIVE):
        s, _ = run_folds(make_model(RNG_SEED + rep), rows, seed=RNG_SEED + rep)
        for m in METRICS:
            scores[m].extend(s[m])
    return {m: np.array(v) for m, v in scores.items()}

def compare(incumbent_scores, candidate_scores, n_rows, metric):
    """Corrected paired t-test on the per-fold differences of one metric (same fold draws for both models). Unrounded: verdict() reads
    these values; display_row() rounds them for the register, so a gap of +0.0004 or a p of 0.0496 is judged as itself, not as 0.000 or 0.050."""
    n_test = n_rows // N_FOLDS; n_train = n_rows - n_test
    diff = candidate_scores[metric] - incumbent_scores[metric]
    t_stat, p_val = compute_corrected_ttest(diff, df=len(diff) - 1, n_train=n_train, n_test=n_test)
    return {f"{metric}_cand": float(candidate_scores[metric].mean()), f"{metric}_inc": float(incumbent_scores[metric].mean()),
            f"{metric}_gap": float(diff.mean()), f"{metric}_sd": float(diff.std(ddof=1)),
            f"{metric}_t": float(t_stat), f"{metric}_p": float(p_val)}

def show_p(p):
    """A p-value to three significant figures — 0.0312, 0.00384, 1.3e-05 — so that any p can be placed against the Bonferroni line (4 dp):
    at 3 dp a p of 0.0038 and one of 0.0044 both printed 0.004, one on each side of the line at the cap (0.0042), and anything under 0.0005 printed 0.000."""
    return f"{p:.3g}"

def display_row(row):
    """The register's display precision — 3 dp; `_t` to 2 dp; `_p` to three significant figures (show_p). The verdict was taken on the
    unrounded values before this is called."""
    def shown(k, v):
        if not isinstance(v, float):
            return v
        if k.endswith("_t"):
            return round(v, 2)
        if k.endswith("_p"):
            return float(show_p(v))
        return round(v, 3)
    return {k: shown(k, v) for k, v in row.items()}

def verdict(row):
    """The 3f rule: promoted only if one metric improves measurably and the other does not fall on average. Read on the unrounded gaps and p."""
    for up, keep in [("recall", "precision"), ("precision", "recall")]:
        if row[f"{up}_p"] < 0.05 and row[f"{up}_gap"] > 0 and row[f"{keep}_gap"] >= 0:
            return f"promoted: {up} up, no {keep} loss"
    return "kept: no measured gain without a loss"


In [ ]:
MAX_TESTED_TRIALS = 6
register, trial_scores = [], {}   # one row per trial; trial_scores keeps every model's per-fold scores for Exercise 9

# Rung 0 is the first incumbent: the Niño-sign rule, the physical prior. Every rung has to earn promotion over it with the paired test;
# if nothing is promoted, 3g tests this rule, and 3h asks whether it showed any skill over the base rate at all.
incumbent_name, incumbent_make = "nino-sign", lambda s: ladder(s)["nino-sign"]
incumbent_scores = trial(incumbent_make, train)
trial_scores[incumbent_name] = incumbent_scores
print(f"first incumbent: {incumbent_name}   fold draws per trial: {N_REPEATS_EFFECTIVE} × {N_FOLDS} folds = {len(incumbent_scores['recall'])} paired scores per metric")

def record_trial(name, make_model, what_changed):
    """Run a candidate against the current incumbent on identical fold draws; append to the register; promote only by the 3f rule.
    Idempotent by candidate name: a call whose name is already in the register runs nothing and is skipped, so the Your-trials cell keeps
    every call it ever held — re-running it re-runs no trial, and Restart & Run All rebuilds every row in order before 3g."""
    global incumbent_name, incumbent_make, incumbent_scores
    if any(r["candidate"] == name for r in register):
        print(f"{name}: already in the register — skipped (re-running this cell runs no trial; Restart & Run All rebuilds every row)")
        return pd.DataFrame(register)
    assert len(register) < MAX_TESTED_TRIALS, "trial budget spent: write up what you have (3h)"
    cand = trial(make_model, train)
    trial_scores[name] = cand
    row = dict(trial=len(register) + 1, candidate=name, incumbent=incumbent_name, what_changed=what_changed)
    for m in METRICS:
        row.update(compare(incumbent_scores, cand, len(train), m))
    row["verdict"] = verdict(row)        # on the unrounded gaps and p (compare); the register shows them rounded (display_row)
    register.append(display_row(row))
    if row["verdict"].startswith("promoted"):
        incumbent_name, incumbent_make, incumbent_scores = name, make_model, cand
    return pd.DataFrame(register)

def bonferroni_line():
    """The multiple-comparisons line the register does NOT apply (3f): 0.05 divided by the looks taken so far — two metrics per tested row — and at the cap."""
    looks = 2 * len(register)
    print(f"Bonferroni line after {len(register)} tested row(s) = {looks} looks: p < {0.05 / looks:.4f}   (at the cap of {MAX_TESTED_TRIALS} rows: p < {0.05 / (2 * MAX_TESTED_TRIALS):.4f}). "
          "A row promoted on a p above this line is promoted under the uncorrected rule only.")

# The ladder climb (3a): the three rungs, each against the incumbent, run together because 3a fixed them before any score was read.
record_trial("logistic", lambda s: ladder(s)["logistic"], "rung 1: logistic on the scaled anomaly, default threshold 0.5")
record_trial("tree", lambda s: ladder(s)["tree"], f"rung 2: tree {TREE_SETTINGS}")
record_trial("boosting", lambda s: ladder(s)["boosting"], f"rung 3: boosting {BOOSTING_SETTINGS}")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 70)
print(f"incumbent after the ladder ({len(register)} trials): {incumbent_name}"); bonferroni_line()
pd.DataFrame(register)


**The example trial.** One change, written down before it runs: logistic's decision threshold, lowered from 0.5 to the designed base rate. It is one `record_trial` call: a short name, a function that builds the candidate from a seed, and one
line saying what changed. Read its row against the Bonferroni line the register cell prints (above the table) — a `p` under 0.05 but above that line is a promotion over the
sign rule, not skill — and know that a `p` just under 0.05 is the kind of lead another fold draw can erase.
 


In [ ]:
record_trial("logistic@base-rate",
             lambda s: FixedThresholdClassifier(ladder(s)["logistic"], threshold=1 - WET_QUANTILE, response_method="predict_proba"),
             "example trial: logistic, decision threshold lowered from 0.5 to the designed base rate")
print(f"incumbent after {len(register)} trials: {incumbent_name}"); bonferroni_line()
pd.DataFrame(register).tail(1)


**Your trials.** The cell below is yours. Write a trial's three lines in Handbook entry 8 first — what changes (one thing), why, which metric you
expect to move — then add one `record_trial` call below the last one, run the cell, and read the new row before you choose the next; the cell's
header comment carries the run discipline and Entry 8 scores it. The cell opens with `FINAL = False`: 3g does not run until you set it to `True` after your last trial, so a Restart & Run All between trials never scores the test members. The budget is six tested rows in all — four are already spent (the three rungs and the example trial), two are yours. Candidates: a decision threshold on another rung, a tree depth, leaf size or
boosting learning rate moved one notch.
 


In [ ]:
# ===== YOUR TRIALS — keep every call in this cell. Add the next call below the last and re-run the cell; calls already in the register are
# skipped (record_trial is idempotent by name), so a re-run runs no trial twice and never raises on the budget. Read the new register row
# before you write the next. The budget is six tested rows IN ALL — four are already spent (the three rungs and the example trial), two are yours. =====
# The pattern (copy it under "your calls", edit the three arguments, run the cell):
#   record_trial("tree-depth2",                                                        # a short name for the candidate
#                lambda s: DecisionTreeClassifier(max_depth=2, min_samples_leaf=20, random_state=s),   # builds it from a seed
#                "tree depth 3 -> 2")                                                  # what changed, in words
# The other pattern — a decision threshold on any rung, the example trial's wrapper (a threshold is a trial, 3e):
#   record_trial("tree@base-rate",
#                lambda s: FixedThresholdClassifier(ladder(s)["tree"], threshold=1 - WET_QUANTILE, response_method="predict_proba"),
#                "tree, decision threshold lowered from 0.5 to the designed base rate")
#
# FINAL — read before you flip it:
#   1. Run every trial you add with FINAL = False and READ its register row first (the cell prints the register each run).
#   2. Set FINAL = True ONLY after every call in this cell has already run once in this kernel — never in the same edit that adds a call.
#   3. Then Kernel → Restart & Run All: it rebuilds every row in order, 3g tests the register's final incumbent once and leaves its run counter
#      at 1, and the register the handbook pastes is the one the executed notebook shows. 3g runs ONLY when FINAL is True, so a run-all
#      between trials never scores the test members.
#   The guard at the bottom of this cell enforces 2: a call that first ran in the same run of this cell that found FINAL = True — in a kernel
#   where this cell had run before — is named in one loud line and FINAL is set back to False, so 3g prints "test not run".
# ---------------------------------------------------------------------------------------------------------------------------------
FINAL = False      # set True ONLY after every trial below has already run once (Restart & Run All between); 3g prints "test not run" until then
YOUR_TRIALS_RUNS = globals().get("YOUR_TRIALS_RUNS", 0) + 1   # this cell's runs in one kernel session (Restart resets it to 1), the device 3g's counter uses
_rows_before = len(register)                                   # the register as this run of the cell found it

# ----- your calls: add each record_trial call below this line, above the guard -----


# ----- the guard (keep it below your calls): FINAL = True with a call that just ran for the first time means the register changed since you read it -----
_new_this_run = [r["candidate"] for r in register[_rows_before:]]
if FINAL and YOUR_TRIALS_RUNS > 1 and _new_this_run:
    print(f"FINAL = True REFUSED: {', '.join(_new_this_run)} ran for the first time in this kernel in the same run of this cell that set FINAL = True "
          f"(run #{YOUR_TRIALS_RUNS} of this cell) — the register changed since you last read it. Read the new row(s) below, then set FINAL = True "
          f"and Kernel → Restart & Run All. FINAL is set back to False for this run: 3g prints 'test not run'.")
    FINAL = False

print(f"incumbent after {len(register)} trials: {incumbent_name}   (tested rows left: {MAX_TESTED_TRIALS - len(register)})   FINAL = {FINAL}"); bonferroni_line()
pd.DataFrame(register)


:::{note}
**Exercise 8 — your trials and the register.** Run your own trials in the *Your trials* cell (up to two: six tested rows in all, four already spent), three lines in the Handbook before each; then paste the register and read every row against the spread and the Bonferroni line. → **Handbook entry 8**
:::

:::{note}
**Exercise 9 — the paired test on a pair you choose.** Run the scaffold cell below on the default pair — the last promotion's incumbent and candidate, on the metric that promotion was earned on (read from its verdict; the cell's first line names it); if nothing was promoted, the scaffold takes the register's last row (the Niño-sign rule against the last candidate) on precision — or a pair and metric you choose, and state whether the lead is a measured difference — and what you keep if it is not. → **Handbook entry 9**
:::


In [ ]:
# ===== EXERCISE 9 — the paired test on a pair you choose: two names from trial_scores, one metric, one call =====
promoted = [r for r in register if r["verdict"].startswith("promoted")]
last = promoted[-1] if promoted else register[-1]                 # default pair: the last promotion (the final incumbent and the rung it replaced)
PAIR_A, PAIR_B = last["incumbent"], last["candidate"]             # edit to any two names printed in the register, e.g. "nino-sign", "tree"
# The metric: the one the default pair's promotion was earned on, read from its verdict ("promoted: <metric> up, no <other> loss");
# precision when nothing was promoted. Set it to the other metric yourself to test the no-loss half, or any metric for a pair you choose.
# The test is RIGHT-tailed (3f): a PAIR_B that LOST prints p → 1 by design, so the verdict below reads the gap's sign as the register's
# verdict() does — a loss is named a LOSS, with its left-tail p, never 'NOT a measured difference'.
promoted_on = last["verdict"].split(":")[1].split()[0] if last["verdict"].startswith("promoted") else None
HAND_METRIC = promoted_on or "precision"
print(f"default pair: register row {last['trial']} ({last['candidate']} vs {last['incumbent']}, verdict '{last['verdict']}') → metric {HAND_METRIC}"
      + ("" if promoted_on else "  (nothing promoted: precision by default)"))

n_test = len(train) // N_FOLDS; n_train = len(train) - n_test    # exactly as compare() sets them
a, b = trial_scores[PAIR_A][HAND_METRIC], trial_scores[PAIR_B][HAND_METRIC]
diff = b - a                                                      # candidate minus incumbent, the register's sign convention
df_hand = len(diff) - 1
t_hand, p_hand = compute_corrected_ttest(diff, df=df_hand, n_train=n_train, n_test=n_test)
p_loss = t_dist.cdf(t_hand, df_hand)                              # the LEFT tail: small when PAIR_B lost — exactly when the right-tailed p_hand reads ≈ 1
print(f"{PAIR_B} − {PAIR_A} on {HAND_METRIC}: {len(diff)} paired fold scores   n_train = {n_train}   n_test = {n_test}")
print(f"means {b.mean():.3f} vs {a.mean():.3f}   gap {diff.mean():+.4f}   per-fold SD of the gap {diff.std(ddof=1):.4f}   "
      f"corrected standard error {corrected_std(diff, n_train, n_test):.4f}  (the spread the test uses)")
if p_hand < 0.05:
    reading = f"p < 0.05: a measured GAIN for {PAIR_B} under the uncorrected rule; read it against the Bonferroni line below"
elif p_loss < 0.05:
    reading = (f"a measured LOSS for {PAIR_B}: {abs(t_hand):.1f} corrected standard errors (left-tail p = {show_p(p_loss)}) — "
               f"the right-tailed p reads {show_p(p_hand)} by design (3f); read the gap's sign, as the register's verdict() does")
else:
    reading = "NOT a measured difference (neither tail under 0.05: the gap reads as run-to-run spread)"
print(f"t = {t_hand:.2f}   p = {show_p(p_hand)}   →  {reading}")
bonferroni_line()   # the line the register prints (3f): a p under 0.05 but above it is a measured difference under the uncorrected rule only.
                    # p prints to three significant figures (show_p, 3f) so it can be placed against this line at any size — 0.0038 and 1.3e-05 alike

### 3g Test

The test guards against O2's *test* failure: *a final number that was tuned toward through repeated use of the same held-out cases.* The test members
were set aside in 2b and have not been touched. The final model is the register's current incumbent — the last promotion, or the Niño-sign rule if nothing
was promoted — fit once on **all** training members and scored **once** here; a second attempt would make this a validation set.
The cell runs only when `FINAL = True` has been set in the *Your trials* cell — until then it prints `test not run` and touches nothing, so a Restart & Run All between trials never scores the test members. When it runs it counts its own runs and prints `test run #N`; `#1` is the only passing value, and Entry 10 says what the counter cannot see. The cell ends with two lines that read the one score against no skill: the
model's precision and AP minus the base rate of the same test winters, each with an interval from resampling whole test members; nothing is
refit to draw it. The reading is taken on the bounds as printed (3 dp): an interval whose printed bound is `+0.000` or `-0.000` spans zero and reads *NOT distinguishable*. The same model can read *above no skill* here and *NOT distinguishable* when Module 3 re-tests it, because Module 3 drops each member's first winter and so tests on different rows — a changed reading there is expected, not an error.
 


In [ ]:
if not FINAL:   # the gate (Exercise 8): the register is final only when the learner says so; until then the test members stay untouched
    print(f"test not run: set FINAL = True in the Your-trials cell after your last 3f trial, then Restart & Run All. "
          f"(register so far: {len(register)} tested rows, incumbent {incumbent_name}; the test members have not been scored)")
else:
    TEST_RUNS = globals().get("TEST_RUNS", 0) + 1       # the counter the rubric reads: a second run of this cell in one session is a second test; Restart & Run All resets it to 1
    X_tr, y_tr = train[FEATURES].values, train["wet"].values   # rebuilt from the current FEATURES, so the fit and the test see the same columns (3f's trial() does the same)
    X_te, y_te = test[FEATURES].values, test["wet"].values
    final = incumbent_make(RNG_SEED).fit(X_tr, y_tr)
    yhat_te = final.predict(X_te)
    cm_te = confusion_matrix(y_te, yhat_te, labels=[0, 1])
    score_te = final.predict_proba(X_te)[:, 1]

    print(f"test run #{TEST_RUNS}")
    print(f"final model: {incumbent_name}   test members: {test.member_id.nunique()}   test winters: {len(y_te)}   base rate: {y_te.mean():.3f}")
    print(f"precision (wet) = {precision_score(y_te, yhat_te, zero_division=0):.3f}   recall (wet) = {recall_score(y_te, yhat_te, zero_division=0):.3f}   "
          f"accuracy = {accuracy_score(y_te, yhat_te):.3f}   AP = {average_precision_score(y_te, score_te):.3f}")
    print(classification_report(y_te, yhat_te, target_names=["dry", "wet"], zero_division=0))
    print("test confusion matrix (rows = true, columns = predicted); the figure below draws the same matrix:")
    print(pd.DataFrame(cm_te, index=["true dry", "true wet"], columns=["pred dry", "pred wet"]))

    # the three rung-0 lines (the two bars and the physical prior) on the same test members, for the write-up
    for name in RUNG0:
        yb = ladder(RNG_SEED)[name].fit(X_tr, y_tr).predict(X_te)
        print(f"{name:11s} precision {precision_score(y_te, yb, zero_division=0):.3f}  recall {recall_score(y_te, yb, zero_division=0):.3f}  accuracy {accuracy_score(y_te, yb):.3f}")

    # Is the tested model's skill distinguishable from none? Precision and AP minus the base rate of the SAME test winters (always-wet's
    # precision; a no-skill ranking's AP). The interval resamples whole test members with replacement, N_BOOT times, from the declared seed;
    # the predictions are fixed, so nothing is refit and the test is still run once.
    N_BOOT = 1000
    boot_rng = np.random.RandomState(RNG_SEED)
    member_rows = {m: np.flatnonzero(test["member_id"].values == m) for m in test["member_id"].unique()}
    names = list(member_rows)
    point = {"precision − base rate": precision_score(y_te, yhat_te, zero_division=0) - y_te.mean(),
             "AP − base rate": average_precision_score(y_te, score_te) - y_te.mean()}
    draws = {k: [] for k in point}
    for _ in range(N_BOOT):
        pick = np.concatenate([member_rows[m] for m in boot_rng.choice(names, size=len(names), replace=True)])
        yb, hb, sb = y_te[pick], yhat_te[pick], score_te[pick]
        draws["precision − base rate"].append(precision_score(yb, hb, zero_division=0) - yb.mean())
        draws["AP − base rate"].append(average_precision_score(yb, sb) - yb.mean())
    for k, v in draws.items():
        lo, hi = (round(float(b), 3) for b in np.percentile(v, [2.5, 97.5]))   # bounds at the PRINTED precision (3 dp): the reading below is decided on exactly what the line shows, so `[+0.000, …]` never reads 'above'
        reading = "above no skill" if lo > 0 else ("below no skill" if hi < 0 else "NOT distinguishable from no skill")
        print(f"{incumbent_name}: {k:22s} {point[k]:+.3f}   95% interval over test members [{lo:+.3f}, {hi:+.3f}]  →  {reading}")

    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    ConfusionMatrixDisplay(cm_te, display_labels=["dry", "wet"]).plot(ax=ax, colorbar=False, cmap=GOLD_CMAP)
    ax.set_xlim(-0.85, 1.85); ax.set_ylim(1.85, -0.85)   # a canvas frame around the matrix, as in 3d
    ax.set_title(f"Test: {incumbent_name} on {test.member_id.nunique()} unseen members", fontsize=9); cite(fig); plt.tight_layout()


:::{note}
**Exercise 10 — the one test.** Paste the test block from its `test run #` line with the two no-skill interval lines and the three rung-0 lines (the two bars and the physical prior) on the same members; did validation predict the test? → **Handbook entry 10**
:::

### 3h Evaluate and justify — your final decision

Unidata's closing instruction (Apache-2.0): "No matter your final decision, the most important
aspect is your ability to justify it using data and a robust evaluation." The *justify* step guards against
O2's last failure: *a model chosen for its sophistication rather than its measured performance and cost.* 

**Your recommendation, to the forecaster drafting the autumn outlook (Part 1),** in three labelled parts, each claim pointing at a number in
your Entries 6–10 (Entry 11's prompt carries each part's questions): (1) *does it deliver?* — the tested model's precision and AP against the base
rate of the same test winters, read with the two interval lines in 3g; (2) *if nothing showed skill, say so plainly* — the simplest answer is then
the base rate itself, the outlook stays at "equal chances", and what the run measured is kept apart from what it cannot attribute; (3) *the next
change, and its cost* — a new season, box or autumn input is one parameter or derivation plus `REBUILD = True` (roughly 160 GB read: 33 minutes on
the course's one full derivation, an hour or more on home broadband); a threshold trial on another rung costs minutes and no rebuild; keeping one half is a filter on the table at the price of half the
members and half the power; a snowfall target (PRECSC + PRECSL) is one more derivation. 

:::{note}
**Exercise 11 — your final decision.** Write the recommendation in the Handbook, three parts labelled, every claim pointing at a number in Entries 6–10. → **Handbook entry 11**
:::


## Summary

The spine was walked end to end on the NSF NCAR-distributed ensemble: a question framed as a decision; a Metadata Document
filled from the catalogue, not memory; the forced response removed and checked by a test it could fail; a split by ensemble member, the transforms
refit inside every fold, and a contrast cell that measured the two leaks here rather than assuming them; a three-rung ladder scored against the
base rate and a one-rule classifier; refinement trials compared as a paired test against the corrected spread, each `p` uncorrected with the
Bonferroni line printed and the looks bounded at six; one test run once and read against no skill; and a decision justified on measured
performance and cost, including when no model beats the simplest answer. The contrast cell's leak gaps came out
near zero here, for reasons the notebook prints: the lag-1 table in 2c shows wet winters barely persist inside a member, so a
member-blind split has little to memorise; the single input carries no member identity; and the transforms are a smooth fit to many
training members' mean and one pooled quantile, so leaving one fold out barely moves them. With a member-specific input, or transforms fit on fewer members, the same cell is where the split earns its keep. 

**What the result does and does not say.** It is a statement about CESM2 at ~1°: the model's own winters, its own ENSO and teleconnection, its own
smoothed mountains. Testing the same question against observed winters (a station or gridded precipitation record against observed
Niño 3.4), or on a finer-resolution ensemble, is the next change that could tell a model result from a real one. 

**The design defaults, as recorded.** Seven design decisions, recorded with their reasons and alternatives in the Metadata Document (2a); each is a parameter or a rule in the code, none is written into any answer: the forced response removed as a
degree-`FORCED_DEG` fit; `wet` against one pooled threshold; test members drawn per half (`TEST_FRACTION`); `N_FOLDS = min(N_FOLDS_MAX, training members)`;
rung 0 is three rules with the Niño-sign rule as first incumbent; promotion on corrected `p < 0.05` with no mean loss in the other metric; and
the 3d table carries fold-mean and pooled precision/recall. `ACCESS_DATE` is the cache header's `access=` field — written by the `REBUILD = True`
run that made the table and asserted equal to it on every run. All of it follows scikit-learn's rule for a rerunnable fit: declare the seed once, pass it down, and let a
`Pipeline` keep every transform on the training side of the split. 


In [ ]:
# The spine as this notebook walked it: eight steps, the section that took each, and the failure the course's O2 table says the step
# guards against — the module's objective O2 in one picture. A course diagram: no CESM2 LENS data is drawn (credit_own, not cite).
import textwrap
SPINE_WALKED = [
    ("framing",  "Part 1",  "a target that does not answer the decision anyone will make"),
    ("data",     "2a · 2b", "inputs whose origin and quality were never recorded"),
    ("split",    "2c",      "training and scoring sets that share ensemble members, so the \"unseen\" score is inflated"),
    ("train",    "3a–3c",   "a fit that cannot be rerun because its settings and seed were not recorded"),
    ("validate", "3d",      "tuning with no held-out score to show whether the model is memorising"),
    ("refine",   "3e · 3f", "stopping at the first configuration tried"),
    ("test",     "3g",      "a final number that was tuned toward through repeated use of the same held-out cases"),
    ("justify",  "3h",      "a model chosen for its sophistication rather than its measured performance and cost"),
]
fig, ax = plt.subplots(figsize=(12, 3.2)); ax.set(xlim=(0, len(SPINE_WALKED)), ylim=(0.22, 1)); ax.axis("off")
for k, (step, where, failure) in enumerate(SPINE_WALKED):
    ax.add_patch(plt.Rectangle((k + 0.06, 0.6), 0.88, 0.32, facecolor="#E7CD82", edgecolor="#8F7022", lw=1.2))
    ax.text(k + 0.5, 0.82, step, ha="center", va="center", fontsize=10.5, fontweight="bold")
    ax.add_patch(plt.Rectangle((k + 0.12, 0.62), 0.76, 0.12, facecolor="#F5F3ED", edgecolor="#E4E0D6", lw=0.8))
    ax.text(k + 0.5, 0.68, where, ha="center", va="center", fontsize=8.5)
    ax.text(k + 0.5, 0.54, textwrap.fill("guards against " + failure, 21), ha="center", va="top", fontsize=6.6, linespacing=1.25)
    if k < len(SPINE_WALKED) - 1:
        ax.annotate("", xy=(k + 1.06, 0.76), xytext=(k + 0.94, 0.76), arrowprops=dict(arrowstyle="->", color="#8F7022", lw=1.3))
ax.set_title("The supervised-learning spine as this notebook walked it — each step, where it was taken, and the failure it guards against (the course's O2 table)", fontsize=9)
credit_own(fig, "a diagram of the course spine (shape: NSF Unidata, Apache-2.0); no CESM2 LENS data drawn"); plt.tight_layout()


### What's next?

Module 3 · Analysis keeps the same data, split and spine, replaces the ladder with a small Keras MLP read through its loss curves, asks what
drove three of its predictions with one attribution method, and ends with an AI weather model and the two conditions under which its output
should not be trusted. 

## Additional resources

### Data and literature

| Resource | Where it bears on this notebook | Licence |
|---|---|---|
| **Data (required citation):** Community Earth System Model v2 Large Ensemble (CESM2 LENS) was accessed on 2026-10-20 from https://registry.opendata.aws/ncar-cesm2-lens. Documentation DOI 10.26024/y48t-q717. Results acknowledge the CESM2 Large Ensemble Community Project and supercomputing resources provided by the IBS Center for Climate Physics in South Korea. | the one data source; the access line under every figure (2a's `ACCESS_DATE`) | Registry of Open Data on AWS; UCAR Data Terms of Use (CC BY 4.0) |
| Grover, M., & Banihirwe, A. (2021). *NCAR CESM2 Large Ensemble Cloud-Optimized Dataset*. NSF NCAR CISL. https://doi.org/10.26024/y48t-q717 | NSF NCAR's recommended dataset citation for the AWS subset, cited with Rodgers et al. 2021 | UCAR Data Terms of Use |
| Rodgers, K. B. et al. (2021). Ubiquity of human-induced changes in climate variability. *Earth System Dynamics* 12, 1393–1411. https://doi.org/10.5194/esd-12-1393-2021 | the ensemble paper: §2.2 is the Window row (2a) and the source of the Niño 3.4 box | CC BY 4.0 |
| Stagge, J. H. et al. (2023). Orographic amplification of El Niño teleconnections on winter precipitation across the Intermountain West. *Nature Water* 1, 1016–1026. https://doi.org/10.1038/s44221-023-00163-9 | the ENSO dipole and the Front Range in its transition zone (Part 1; Entry 2) | CC BY 4.0 |
| NWS Boulder (2020). La Niña and the Upcoming 2020-2021 Winter Season. https://www.weather.gov/media/bou/LaNinaWinterOutlook_Sept20.pdf | the expected sign east of the Divide (Entry 1); the physical prior rung 0 carries | US Government |
| NWS Pueblo (2025). NWS Pueblo Weather Update: Winter of 2025-2026 Outlook (slide deck). https://www.weather.gov/media/pub/pdf/SC_SE_CO_Winter_2025_26_Outlook.pdf | PNA and NAO as the patterns with the highest correlations (3b); the decks NWS Boulder and Pueblo issue (Part 1) | US Government work, public domain |

### Software, templates and the spine

| Resource | Where it bears on this notebook | Licence |
|---|---|---|
| NSF Unidata, *Supervised Machine Learning Readiness* (NSF award #2319979). https://github.com/Unidata/supervised-ml-readiness | the spine, handbook and rubric shape | Apache-2.0 |
| scikit-learn User Guide: §3.1 Cross-validation · §3.4 Metrics and scoring · §12 Common pitfalls · example "Statistical comparison of models using grid search". https://scikit-learn.org/stable/ | grouped splits (2c), precision/recall/AP (3d), the leakage rule (Entry 5), the corrected paired t (3f) | BSD-3-Clause |
| Project Pythia notebook template. https://foundations.projectpythia.org/appendix/template | the front-matter and closing-section shape of this notebook | CC BY 4.0 |

*This module was self-funded and has not been reviewed or sponsored by NSF NCAR, COMET or NSF Unidata. Content CC BY 4.0; code Apache-2.0.*
